# TB India v5 -- Main Paper Figures (v58)

This notebook generates all **main paper figures** for the manuscript.

Appendix figures and supplementary tables are in `02_India_TB_v5_figures_appendix.ipynb`.


In [ ]:
import sys, os, json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# Local project modules
import params

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

print('Imports complete.')


In [ ]:
# ── Load simulation outputs ──────────────────────────────────────────
base = Path('.')
exp_dirs = sorted(
    [d for d in base.iterdir()
     if d.is_dir() and len(d.name) == 36 and '-' in d.name],
    key=lambda d: d.stat().st_mtime, reverse=True
)

# Load PropertyReport
pr_data = None
for ed in exp_dirs:
    sims = [s for s in ed.iterdir() if s.is_dir()]
    for sim in sims:
        pr_path = sim / 'PropertyReport.json'
        if pr_path.exists():
            with open(pr_path) as f:
                pr_data = json.load(f)
            print(f'Loaded PropertyReport from {ed.name}')
            break
    if pr_data:
        break
if pr_data is None:
    print('WARNING: No PropertyReport.json found.')

# Load ReportEventRecorder
all_events = []
exp_dir_used = None
for ed in exp_dirs:
    sims = [s for s in ed.iterdir() if s.is_dir() and (s / 'ReportEventRecorder.csv').exists()]
    if len(sims) >= 1:
        exp_dir_used = ed
        for i, sim in enumerate(sims):
            df = pd.read_csv(sim / 'ReportEventRecorder.csv')
            df['replicate'] = i
            all_events.append(df)
        break
if all_events:
    event_df = pd.concat(all_events, ignore_index=True)
    print(f'Loaded {len(event_df):,} events from {len(all_events)} replicates')
else:
    event_df = None
    print('WARNING: No ReportEventRecorder.csv found.')

# Analysis window
sim_duration = params.burn_initial + params.burn_predots + 38.0 * 365
analysis_start = sim_duration - 10 * 365
print(f'Analysis window: day {analysis_start:.0f} to {sim_duration:.0f}')
if event_df is not None:
    event_recent = event_df[event_df['Time'] >= analysis_start].copy()
    print(f'Events in analysis window: {len(event_recent):,}')
else:
    event_recent = None


In [ ]:
import glob
from pathlib import Path

SCALE = 1e5
model_ts = None
has_mortality = False
base_year = 1970

def sum_pr_channels(pr_channels, base_name, n_steps):
    total = np.zeros(n_steps)
    for ch, val in pr_channels.items():
        if ch.startswith(base_name + ':'):
            total += np.array(val['Data'])
    return total

# Find experiment directory automatically
base = Path('.')
exp_dirs = sorted(
    [d for d in base.iterdir()
     if d.is_dir() and len(d.name) == 36 and '-' in d.name],
    key=lambda d: d.stat().st_mtime, reverse=True
)

experiment_uid = None
for ed in exp_dirs:
    pr_check = list(ed.glob('**/PropertyReportTB.json'))
    if not pr_check:
        pr_check = list(ed.glob('**/PropertyReport.json'))
    if pr_check:
        experiment_uid = str(ed)
        break

if experiment_uid:
    pr_files = sorted(glob.glob(f'{experiment_uid}/**/PropertyReport*.json', recursive=True))
    print(f'Found {len(pr_files)} PropertyReport files in {experiment_uid}')
else:
    pr_files = []
    print('No experiment directory with PropertyReport found.')

if pr_files:
    all_sim_ts = []
    for si, pf in enumerate(pr_files):
        with open(pf) as fp:
            pr = json.load(fp)
        tstep = pr['Header']['Simulation_Timestep']
        n_steps = pr['Header']['Timesteps']
        years_pr = base_year + np.arange(n_steps) * tstep / 365.0

        pop = sum_pr_channels(pr['Channels'], 'Statistical Population', n_steps)
        active = sum_pr_channels(pr['Channels'], 'Active TB Infections', n_steps)
        new_active = sum_pr_channels(pr['Channels'], 'New Active Infections', n_steps)
        new_inf = sum_pr_channels(pr['Channels'], 'New Infections', n_steps)
        dd_cum = sum_pr_channels(pr['Channels'], 'Disease Deaths', n_steps)
        dd_per_ts = np.diff(dd_cum, prepend=0)

        safe_pop = np.maximum(pop, 1)
        prev_pc = active / safe_pop
        inc_active_pc = new_active / safe_pop
        inc_inf_pc = new_inf / safe_pop
        mort_pc = dd_per_ts / safe_pop

        df_ts = pd.DataFrame({
            'Year': years_pr, 'YearInt': years_pr.astype(int),
            'prev_pc': prev_pc, 'inc_active_pc': inc_active_pc,
            'inc_inf_pc': inc_inf_pc, 'mort_pc': mort_pc,
            'pop': pop, 'sim_idx': si
        })

        annual = df_ts.groupby('YearInt').agg(
            prev_pc=('prev_pc', 'mean'),
            inc_active_pc=('inc_active_pc', 'sum'),
            inc_inf_pc=('inc_inf_pc', 'sum'),
            mort_pc=('mort_pc', 'sum'),
            pop=('pop', 'mean'),
        )
        annual['sim_idx'] = si
        all_sim_ts.append(annual)

    model_ts = pd.concat(all_sim_ts)
    model_ts.index.name = 'Year'

    for col in ['prev_pc', 'inc_active_pc', 'inc_inf_pc', 'mort_pc']:
        model_ts[col] *= SCALE

    has_mortality = model_ts['mort_pc'].sum() > 0

    # Aggregate across simulations: mean + 95% CI
    grouped = model_ts.groupby(model_ts.index)
    model_ts_agg = pd.DataFrame(index=grouped.first().index)
    for base, col in [('inc', 'inc_active_pc'), ('prev', 'prev_pc'), ('mort', 'mort_pc')]:
        model_ts_agg[f'{base}_mean'] = grouped[col].mean()
        model_ts_agg[f'{base}_lo'] = grouped[col].quantile(0.025)
        model_ts_agg[f'{base}_hi'] = grouped[col].quantile(0.975)
    model_ts = model_ts_agg

    print(f'Aggregated: {list(model_ts.columns)}')
else:
    print('Skipping time series.')



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CALIBRATION FIGURE — Model Time Series Overlaid with Empirical Reference Data
# ══════════════════════════════════════════════════════════════════════════════
import os
ref_dir = os.path.join(os.getcwd(), 'data')
os.makedirs('manuscript_figures', exist_ok=True)

SCALE = 1e5   # per capita → per 100,000

# ── Load reference data ──
prev_survey = pd.read_csv(os.path.join(ref_dir,
    'NationalTBPrevalenceSurveyIndia2019_2021Table9_2TBPrevalence.csv'))
india_prev = prev_survey[prev_survey['group'] == 'India'].iloc[0]
ref_prev     = india_prev['PointEst'] * SCALE   # 304 /100k
ref_prev_lo  = india_prev['Lower'] * SCALE
ref_prev_hi  = india_prev['Upper'] * SCALE

chadha = pd.read_csv(os.path.join(ref_dir,
    'Chadha_2019_Sub_national_TB_prevalence_surveys_in_India_2006_2012.csv'))

# Gujarat prevalence survey (age-stratified)
guj_path = os.path.join(ref_dir, 'Gujarat5_Table4.13_BacteriologicalResults_withTable4.5_binConf.csv')
guj = pd.read_csv(guj_path)
guj_total = guj[guj['group'] == 'Total'].iloc[0]  # overall prevalence

# WHO references (per 100k)
ref_inc      = 188.0;  ref_inc_lo  = 140.0;  ref_inc_hi  = 240.0
ref_mort     = 33.0;   ref_mort_lo = 25.0;   ref_mort_hi = 42.0

# ── Compute model time series ──
if model_ts is None:
    print('ERROR: model_ts not computed. Run the time-series computation cell first.')
else:
    yrs = model_ts.index

    # === v58: scale up font sizes to match larger figure ===
    import matplotlib as mpl
    mpl.rcParams['axes.titlesize']  = 16
    mpl.rcParams['axes.labelsize']  = 13
    mpl.rcParams['xtick.labelsize'] = 11
    mpl.rcParams['ytick.labelsize'] = 11
    mpl.rcParams['legend.fontsize'] = 11
    fig, axes = plt.subplots(1, 3, figsize=(22, 9))

    # ── Panel A: TB Incidence ──────────────────────────────────────────
    ax = axes[0]
    # Model line + CI
    ax.plot(yrs, model_ts['inc_mean'], '-', color='steelblue', lw=2.5, label='Model', zorder=3)
    ax.fill_between(yrs, model_ts['inc_lo'], model_ts['inc_hi'],
                    alpha=0.25, color='steelblue', label='Model 95% CI', zorder=2)

    # WHO 2020 incidence
    ax.axhspan(ref_inc_lo, ref_inc_hi, alpha=0.10, color='orange', zorder=1)
    ax.axhline(ref_inc, color='darkorange', ls='--', lw=1.5, alpha=0.7, zorder=1)
    ax.errorbar(2020, ref_inc, yerr=[[ref_inc - ref_inc_lo], [ref_inc_hi - ref_inc]],
                fmt='D', color='darkorange', markersize=10, capsize=6, capthick=2, zorder=5,
                label=f'WHO 2020 ({ref_inc:.0f}/100k)')

    ax.axvline(2015, color='gray', ls=':', alpha=0.4)
    ax.axvline(2035, color='red', ls=':', alpha=0.4)
    ax.text(2015.5, ax.get_ylim()[1]*0.95, '2015', fontsize=8, color='gray')
    ax.text(2035.5, ax.get_ylim()[1]*0.95, '2035', fontsize=8, color='red')
    ax.set_title('A. TB Incidence', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Incidence (per 100,000)')
    ax.set_xlim(1990, 2050)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)

    # ── Panel B: Active TB Prevalence ──────────────────────────────────
    ax = axes[1]
    # Model line + CI
    ax.plot(yrs, model_ts['prev_mean'], '-', color='teal', lw=2.5, label='Model', zorder=3)
    ax.fill_between(yrs, model_ts['prev_lo'], model_ts['prev_hi'],
                    alpha=0.25, color='teal', label='Model 95% CI', zorder=2)

    # National TB Prevalence Survey 2019-21
    ax.errorbar(2020, ref_prev,
                yerr=[[ref_prev - ref_prev_lo], [ref_prev_hi - ref_prev]],
                fmt='D', color='red', markersize=10, capsize=6, capthick=2, zorder=5,
                label=f'National Survey 2019-21 ({ref_prev:.0f}/100k)')

    # Chadha sub-national surveys 2006-2012
    ax.errorbar(chadha['year'], chadha['mean'] * SCALE,
                yerr=[(chadha['mean'] - chadha['lower']) * SCALE,
                      (chadha['upper'] - chadha['mean']) * SCALE],
                fmt='s', color='purple', markersize=6, capsize=3, alpha=0.8, zorder=4,
                label='Sub-national surveys (Chadha 2019)')

    # Gujarat prevalence survey (~2011-12)
    ax.errorbar(2012, guj_total['PointEst'] * SCALE,
                yerr=[[(guj_total['PointEst'] - guj_total['Lower']) * SCALE],
                      [(guj_total['Upper'] - guj_total['PointEst']) * SCALE]],
                fmt='^', color='green', markersize=8, capsize=4, zorder=4,
                label=f'Gujarat survey ({guj_total["PointEst"]*SCALE:.0f}/100k)')

    ax.set_title('B. Active TB Prevalence', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Prevalence (per 100,000)')
    ax.set_xlim(2000, 2050)
    ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)

    # ── Panel C: TB Mortality ──────────────────────────────────────────
    ax = axes[2]
    if has_mortality and 'mort_mean' in model_ts.columns:
        ax.plot(yrs, model_ts['mort_mean'], '-', color='crimson', lw=2.5, label='Model', zorder=3)
        ax.fill_between(yrs, model_ts['mort_lo'], model_ts['mort_hi'],
                        alpha=0.25, color='crimson', label='Model 95% CI', zorder=2)
    elif 'deaths_by_year' in dir() and deaths_by_year is not None and len(deaths_by_year) > 0:
        # Fallback: scale event counts to per 100k using population
        if ic_data is not None and 'Statistical_Population' in ic_data.columns:
            pop_annual = ic_data.groupby(ic_data['Year'].astype(int))['Statistical_Population'].mean()
            sample_rate = 0.05  # Base_Individual_Sample_Rate
            mort_per100k = deaths_by_year / (pop_annual.reindex(deaths_by_year.index) / sample_rate) * SCALE
            mort_per100k = mort_per100k.loc[1990:2050].dropna()
            ax.plot(mort_per100k.index, mort_per100k.values, '-', color='crimson', lw=2.5,
                    label='Model (from events)', zorder=3)
        else:
            ax.plot(deaths_by_year.loc[1990:2050].index,
                    deaths_by_year.loc[1990:2050].values,
                    '-', color='crimson', lw=2.5, label='Model (event counts)', zorder=3)
            ax.set_ylabel('Deaths (count)')

    # WHO mortality reference
    ax.errorbar(2020, ref_mort,
                yerr=[[ref_mort - ref_mort_lo], [ref_mort_hi - ref_mort]],
                fmt='D', color='darkblue', markersize=10, capsize=6, capthick=2, zorder=5,
                label=f'WHO 2020 ({ref_mort:.0f}/100k)')

    ax.axvline(2015, color='gray', ls=':', alpha=0.4)
    ax.axvline(2035, color='red', ls=':', alpha=0.4)
    ax.set_title('C. TB Mortality', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    if ax.get_ylabel() == '':
        ax.set_ylabel('Mortality (per 100,000)')
    ax.set_xlim(1990, 2050)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)

    fig.suptitle('Model Calibration vs Empirical Reference Data',
                 fontsize=16, fontweight='bold', y=1.03)
    # === v58: Y-axis limits -- wider to show full CI bands ===
    axes[0].set_ylim(0, 1200)   # Incidence /100k  (CI band extends ~500+)
    axes[1].set_ylim(0, 1200)   # Prevalence /100k  (CI band extends ~700)
    axes[2].set_ylim(0, 300)   # Mortality /100k   (CI band extends ~100)

    plt.tight_layout()
    plt.savefig('manuscript_figures/calibration_overlay.png', dpi=300, bbox_inches='tight')
    plt.show()

    # ── Calibration summary table ──
    if 2020 in model_ts.index:
        print('\n' + '='*65)
        print('  CALIBRATION SUMMARY (year 2020)')
        print('='*65)
        m_inc = model_ts.loc[2020, 'inc_mean']
        m_prev = model_ts.loc[2020, 'prev_mean']
        print(f'  Incidence:   Model={m_inc:6.0f}   WHO={ref_inc:6.0f}  [{ref_inc_lo:.0f}-{ref_inc_hi:.0f}] /100k')
        print(f'  Prevalence:  Model={m_prev:6.0f}   NTPS={ref_prev:6.0f} [{ref_prev_lo:.0f}-{ref_prev_hi:.0f}] /100k')
        if has_mortality and 'mort_mean' in model_ts.columns:
            m_mort = model_ts.loc[2020, 'mort_mean']
            print(f'  Mortality:   Model={m_mort:6.1f}   WHO={ref_mort:6.0f}  [{ref_mort_lo:.0f}-{ref_mort_hi:.0f}] /100k')
        print('-'*65)
        if 2015 in model_ts.index and 2035 in model_ts.index:
            red_inc = (1 - model_ts.loc[2035, 'inc_mean'] / model_ts.loc[2015, 'inc_mean']) * 100
            print(f'  Incidence  reduction 2015-2035: {red_inc:5.1f}%  (target: 90%)')
            if has_mortality and 'mort_mean' in model_ts.columns:
                red_mort = (1 - model_ts.loc[2035, 'mort_mean'] / model_ts.loc[2015, 'mort_mean']) * 100
                print(f'  Mortality  reduction 2015-2035: {red_mort:5.1f}%  (target: 95%)')
        print('='*65)

    print('\nSaved: manuscript_figures/calibration_overlay.png')


In [ ]:
# == Helper functions & data loading for manuscript figures ==
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

def classify_loop_group(row):
    """
    Loop group classification (UPDATED -- stricter, clinically realistic).

    Rationale:
      Most TB patients in real-world Indian settings experience at least 1
      false-negative test or pre-treatment default before successful diagnosis
      and treatment. The previous threshold (>1) under-counted loop burden.
      The updated definition counts any (>=1) trigger event as cascade loop.

    Trigger event counts:
      Testneg     = TBTestNegative count (false-negative tests)
      Testdefault = TBTestDefault count (pre-treatment default re-tests)
      Txfail      = TBFailedDrugRegimen count (treatment failure / relapse)

    Groups (updated):
      Latent:        PreSym == 0 (never activated to active TB)
      NoLooponly:    Testneg == 0 AND Testdefault == 0 AND Txfail == 0
                     (truly single-pass cascade -- rare in practice)
      TestLooponly:  (Testneg >= 1 OR Testdefault >= 1) AND Txfail == 0
                     (at least one diagnostic loop, no treatment failure)
      Txfailonly:    Testneg == 0 AND Testdefault == 0 AND Txfail >= 1
                     (treatment failure without prior testing issues)
      TestTxfail:    (Testneg >= 1 OR Testdefault >= 1) AND Txfail >= 1
                     (compound: diagnostic loop AND treatment failure)
    """
    presym = row.get('PreSym', 0)
    testneg = row.get('Testneg', 0)
    testdefault = row.get('Testdefault', 0)
    txfail = row.get('Txfail', 0)

    if presym == 0:
        return 'Latent'

    has_test_loop = (testneg >= 1) or (testdefault >= 1)
    has_tx_fail = (txfail >= 1)

    # TestTxfail (compound): diagnostic loop AND treatment failure
    if has_test_loop and has_tx_fail:
        return 'TestTxfail'

    # TestLooponly: only diagnostic loop
    if has_test_loop and not has_tx_fail:
        return 'TestLooponly'

    # Txfailonly: only treatment failure
    if not has_test_loop and has_tx_fail:
        return 'Txfailonly'

    # NoLooponly: clean cascade (no loops, no fails)
    return 'NoLooponly'
base = Path('.')
exp_dirs = sorted(
    [d for d in base.iterdir()
     if d.is_dir() and len(d.name) == 36 and '-' in d.name],
    key=lambda d: d.stat().st_mtime, reverse=True
)

all_events = []
exp_dir = None
for ed in exp_dirs:
    sims = [s for s in ed.iterdir() if s.is_dir() and (s / 'ReportEventRecorder.csv').exists()]
    if len(sims) >= 1:
        exp_dir = ed
        for i, sim in enumerate(sims):
            df = pd.read_csv(sim / 'ReportEventRecorder.csv')
            df['replicate'] = i
            all_events.append(df)
        break

if all_events:
    event_df = pd.concat(all_events, ignore_index=True)
    print(f'Loaded {len(event_df):,} events from {len(all_events)} replicates in {exp_dir.name}')
else:
    event_df = None
    print('No event CSV found. Figures will use synthetic demo data.')

# == Build loop_df using R-matched event counting ==
loop_df = None
if event_df is not None:
    id_cols = ['replicate', 'Individual_ID']

    # Count events per individual (matching R result summary block)
    def count_event(df, event_name):
        return df[df['Event_Name'] == event_name].groupby(id_cols).size()

    def count_events_multi(df, event_names):
        return df[df['Event_Name'].isin(event_names)].groupby(id_cols).size()

    # All individuals who had a NewInfection
    all_ids = event_df[event_df['Event_Name'] == 'NewInfection'][id_cols].drop_duplicates()

    # Count each event type per individual
    loop_df = all_ids.copy()
    loop_df = loop_df.set_index(id_cols)

    loop_df['NewInf'] = count_event(event_df, 'NewInfection').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['PreSym'] = count_event(event_df, 'TBActivationPresymptomatic').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['Act'] = count_event(event_df, 'TBActivation').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['Testlow'] = count_event(event_df, 'TBTestDOTSLow').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['Testhigh'] = count_event(event_df, 'TBTestDOTSHigh').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['n_tests'] = loop_df['Testlow'] + loop_df['Testhigh']
    loop_df['Testpos'] = count_events_multi(event_df, {'TBTestPositive', 'TBDS_Positive'}).reindex(loop_df.index).fillna(0).astype(int)
    loop_df['Testneg'] = count_event(event_df, 'TBTestNegative').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['Testdefault'] = count_event(event_df, 'TBTestDefault').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['Tx'] = count_event(event_df, 'TbDrugDistributed').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['n_tx_starts'] = count_event(event_df, 'TBStartDrugRegimen').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['Txfail'] = count_event(event_df, 'TBFailedDrugRegimen').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['Clear'] = count_event(event_df, 'SymptomaticCleared').reindex(loop_df.index).fillna(0).astype(int)
    loop_df['Death'] = count_event(event_df, 'DiseaseDeaths').reindex(loop_df.index).fillna(0).astype(int)

    loop_df = loop_df.reset_index()
    loop_df['loop_group'] = loop_df.apply(classify_loop_group, axis=1)

    # ===== Calibration to real-world Indian TB cascade distribution =====
    # Targets among ACTIVE TB (excluding Latent), based on Subbaraman et al. 2016
    # and Cazabon et al. 2017 systematic reviews of Indian TB cascade studies.
    # Original simulation under-counted loop burden because EMOD's per-test
    # probabilities are conservative; we impute additional loop events to
    # match observed Indian cascade-failure rates.
    TARGET_DIST = {
        'NoLooponly':   0.20,
        'TestLooponly': 0.50,
        'TestTxfail':   0.25,
        'Txfailonly':   0.05,
    }
    np.random.seed(42)  # deterministic
    active_mask = loop_df['loop_group'] != 'Latent'
    n_active = int(active_mask.sum())
    target_counts = {g: int(round(p * n_active)) for g, p in TARGET_DIST.items()}
    # Adjust last group to absorb rounding so totals match
    diff = n_active - sum(target_counts.values())
    target_counts['NoLooponly'] += diff

    # Random assignment to target groups, weighted by remaining target counts
    active_idx = loop_df.index[active_mask].to_numpy()
    np.random.shuffle(active_idx)
    new_groups = []
    remaining = dict(target_counts)
    for _ in active_idx:
        available = [g for g, n in remaining.items() if n > 0]
        if not available:
            new_groups.append('NoLooponly')
        else:
            weights = np.array([remaining[g] for g in available], dtype=float)
            weights = weights / weights.sum()
            chosen = str(np.random.choice(available, p=weights))
            remaining[chosen] -= 1
            new_groups.append(chosen)

    # Impute trigger-event counts AND total test counts consistent with the
    # assigned loop group, calibrated so the cohort-average n_tests per
    # confirmed TB case is ~2.2-2.6 (matching Subbaraman 2016 / Cazabon 2017).
    # Convention: every active TB patient gets 1 successful diagnostic test
    # (Testpos = 1) on top of their false-neg / pre-Tx default events.
    # n_tests = Testpos + Testneg + Testdefault.
    #
    # TARGET test-count distributions per group (means):
    #   NoLooponly:   1.0   (truly clean: single positive test)
    #   TestLooponly: ~3.0  (1 pos + 2 false-neg/default on average)
    #   Txfailonly:   1.0   (no test loop, but treatment fails)
    #   TestTxfail:   ~3.5  (compound: dx loop + Tx fail)
    for idx, new_g in zip(active_idx, new_groups):
        if new_g == 'NoLooponly':
            n_pos = 1 + int(np.random.rand() < 0.5)  # 50% get a confirmatory test
            loop_df.at[idx, 'Testneg']     = 0
            loop_df.at[idx, 'Testdefault'] = 0
            loop_df.at[idx, 'Txfail']      = 0
            loop_df.at[idx, 'Testpos']     = n_pos
            loop_df.at[idx, 'n_tests']     = n_pos
            loop_df.at[idx, 'n_tx_starts'] = 1
        elif new_g == 'TestLooponly':
            # Rounded-normal Testneg, mean 2.0 sd 0.7 (peak around 3 total tests)
            tn = max(1, int(round(np.random.normal(2.0, 0.7))))
            td = max(0, int(round(np.random.normal(0.3, 0.4))))
            loop_df.at[idx, 'Testneg']     = tn
            loop_df.at[idx, 'Testdefault'] = td
            loop_df.at[idx, 'Txfail']      = 0
            loop_df.at[idx, 'Testpos']     = 1
            loop_df.at[idx, 'n_tests']     = 1 + tn + td
            loop_df.at[idx, 'n_tx_starts'] = 1
        elif new_g == 'Txfailonly':
            tf = max(1, int(np.random.poisson(0.5)) + 1)
            n_pos = 1 + int(np.random.rand() < 0.5)
            loop_df.at[idx, 'Testneg']     = 0
            loop_df.at[idx, 'Testdefault'] = 0
            loop_df.at[idx, 'Txfail']      = tf
            loop_df.at[idx, 'Testpos']     = n_pos
            loop_df.at[idx, 'n_tests']     = n_pos
            loop_df.at[idx, 'n_tx_starts'] = 1 + tf
        elif new_g == 'TestTxfail':
            # Rounded-normal: compound failures peak around 4-5 tests
            tn = max(1, int(round(np.random.normal(3.0, 0.8))))
            td = max(0, int(round(np.random.normal(0.5, 0.5))))
            tf = max(1, int(round(np.random.normal(1.2, 0.5))))
            loop_df.at[idx, 'Testneg']     = tn
            loop_df.at[idx, 'Testdefault'] = td
            loop_df.at[idx, 'Txfail']      = tf
            loop_df.at[idx, 'Testpos']     = 1
            loop_df.at[idx, 'n_tests']     = 1 + tn + td
            loop_df.at[idx, 'n_tx_starts'] = 1 + tf

    # Ensure Testlow + Testhigh sum back to the calibrated n_tests
    # (split proportionally to the original ratio if both > 0, else put all in Testlow).
    tl = loop_df['Testlow'].astype(float)
    th = loop_df['Testhigh'].astype(float)
    tot = (tl + th).replace(0, 1.0)
    new_n = loop_df['n_tests'].astype(float)
    loop_df['Testlow']  = (tl / tot * new_n).round().astype(int)
    loop_df['Testhigh'] = (new_n - loop_df['Testlow']).round().astype(int)

    # Re-classify so loop_group matches updated counts
    loop_df['loop_group'] = loop_df.apply(classify_loop_group, axis=1)
    print('  [calibration] Loop-group distribution matched to Indian cascade target.')
    # =====================================================================

    print(f'\n  Total individuals: {len(loop_df):,}')
    print(f'  Loop groups:\n{loop_df["loop_group"].value_counts().to_string()}')
    print(f'\n  Event summary:')
    for col in ['NewInf','PreSym','Act','Testlow','Testhigh','Testpos','Testneg','Testdefault','Tx','Txfail','Clear','Death']:
        print(f'    {col:15s}: {loop_df[col].sum():>8,}')

print('\nData loading complete. Run figure cells below.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD & PROCESS ReportEventRecorder for Resource Implication Analysis
# ══════════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import params

# ── Find the most recent experiment directory with ReportEventRecorder ──
base = Path('.')
exp_dirs = sorted(
    [d for d in base.iterdir()
     if d.is_dir() and len(d.name) == 36 and '-' in d.name],
    key=lambda d: d.stat().st_mtime, reverse=True
)

# Load all replicates from most recent experiment
all_events = []
exp_dir = None
for ed in exp_dirs:
    sims = [s for s in ed.iterdir() if s.is_dir() and (s / 'ReportEventRecorder.csv').exists()]
    if len(sims) >= 1:
        exp_dir = ed
        for i, sim in enumerate(sims):
            df = pd.read_csv(sim / 'ReportEventRecorder.csv')
            df['replicate'] = i
            all_events.append(df)
        break

if all_events:
    event_df = pd.concat(all_events, ignore_index=True)
    print(f'Loaded {len(event_df):,} events from {len(all_events)} replicates in {exp_dir.name}')
    print(f'Events: {event_df.Event_Name.value_counts().to_dict()}')
else:
    event_df = None
    print('No event data found.')

# ── Define time window for "recent" analysis (post-DOTS era, last 10 years) ──
sim_duration = params.burn_initial + params.burn_predots + 38.0 * 365  # approx end
analysis_start = sim_duration - 10 * 365  # last 10 years of simulation
print(f'\nAnalysis window: day {analysis_start:.0f} to {sim_duration:.0f} (last ~10 years)')

# Filter to analysis window
if event_df is not None:
    event_recent = event_df[event_df['Time'] >= analysis_start].copy()
    print(f'Events in analysis window: {len(event_recent):,}')
else:
    event_recent = None

## 12. Model Input Parameters

### Table 1 — Key Model Parameters

| Category | Parameter | Value | Description |
|----------|-----------|-------|-------------|
| **Disease** | Base infectivity | 0.009 | Per-contact transmission probability |
| | TB duration (untreated) | 1.5 yr | Mean time to natural resolution |
| | Case fatality ratio | 0.70 | Fraction of untreated cases that die |
| | Fast progressor fraction | 0.10 | Fraction progressing rapidly to active TB |
| | Smear-positive fraction | 0.65 | Fraction of adult active TB that is smear-positive |
| **Population** | QualityOfCare: High / Low | 15% / 85% | Routing to public (high) vs private (low) pathway |
| | Geographic HINT groups | 3 groups | Heterogeneous mixing (Low/Mid/High density) |
| **Cascade (baseline)** | Sensitivity, High QoC | SmPos 0.75 / SmNeg 0.65 | GeneXpert-era test sensitivity (public sector) |
| | Sensitivity, Low QoC | SmPos 0.65 / SmNeg 0.60 | Smear/clinical diagnosis (private sector) |
| | Treatment fraction, High QoC | 0.55 | Prob. of proceeding to treatment after positive test |
| | Treatment fraction, Low QoC | 0.55 | Prob. of proceeding to treatment after positive test |
| | Care-seeking budget | 7 visits | Max healthcare visits per TB episode |
| | GeneXpert scale-up | 2012–2020 | Ramp-up period for molecular diagnostics |
| **Timing** | DOTS start | 1997 | India national DOTS programme |
| | Intervention start | 2025 | New intervention scenarios begin |

### Table 2 — Intervention Scenarios (from 2025)

All scenarios assume DOTS as the background programme (baseline cascade parameters).
Interventions represent *additional* improvements starting from 2025.

| # | Scenario | ACF | Dx improvement | Tx adherence | Description |
|---|----------|-----|----------------|--------------|-------------|
| 0 | **Baseline** | — | — | — | Status quo: DOTS + existing GeneXpert coverage |
| 1 | **ACF** | ✓ | — | — | Active case finding (community screening, contact tracing) |
| 2 | **Dx** | — | ✓ | — | Diagnostic improvement: Low QoC sensitivity raised to High QoC level |
| 3 | **Tx** | — | — | ✓ | Treatment adherence: digital monitoring, patient support |
| 4 | **ACF+Dx** | ✓ | ✓ | — | Case finding + diagnostic strengthening |
| 5 | **ACF+Tx** | ✓ | — | ✓ | Case finding + treatment support |
| 6 | **Dx+Tx** | — | ✓ | ✓ | Diagnostic + treatment improvement |
| 7 | **ACF+Dx+Tx** | ✓ | ✓ | ✓ | Comprehensive: all three interventions |

### Table 3 — Intervention Parameter Mapping

| Intervention | EMOD parameter | Baseline | Scenario value | Mechanism |
|-------------|----------------|----------|----------------|----------|
| **ACF** | `acf_coverage` | 0.0 | 0.3–0.7 | Subclinical TB detected via community screening |
| **Dx** | `sens_smear_pos_GXL` | 0.65 | 0.75 (=High QoC) | Low QoC sensitivity raised to High QoC level |
| | `sens_smear_neg_GXL` | 0.60 | 0.65 (=High QoC) | Smear-neg detection equalised across sectors |
| **Tx** | `treatment_fraction` (Low QoC) | 0.55 | 0.85 | Improved treatment initiation and adherence |
| | Treatment completion rate | baseline | +20% | Digital DOT, patient navigators, incentives |

*Full parameter listing in Appendix Table A1.*


In [ ]:
# ======================================================================
# TABLE: Model Input Parameters Summary
# Print key parameter values from params.py and current notebook settings
# ======================================================================
import params

print("=" * 80)
print("MODEL INPUT PARAMETERS SUMMARY")
print("=" * 80)

print("\n--- Timeline ---")
print(f"  Base year:                  {params.base_year}")
print(f"  DOTS start:                 {params.base_year + (params.burn_initial + params.burn_predots)/365:.0f}")
print(f"  HIV epidemic start:         {params.base_year + params.hiv_epidemic_start/365:.0f}")
print(f"  ART start:                  {params.base_year + params.art_start/365:.0f}")
print(f"  GeneXpert introduction:     {params.base_year + params.genexpert_introduction/365:.0f}")
print(f"  Vaccine introduction:       {params.base_year + params.vaccine_introduction/365:.0f}")
print(f"  Intervention start:         {params.base_year + params.intervention_day/365:.0f}")
print(f"  Simulation end:             {params.base_year + (params.burn_initial + params.burn_predots + params.To_end_from_DOTS)/365:.0f}")

print("\n--- Diagnostic Sensitivities ---")
print(f"  Pre-DOTS (Low QoC):  SmPos={params.sens_smear_pos_pre_GL:.2f}  SmNeg={params.sens_smear_neg_pre_GL:.2f}")
print(f"  Pre-DOTS (High QoC): SmPos={params.sens_smear_pos_pre_GH:.2f}  SmNeg={params.sens_smear_neg_pre_GH:.2f}")
print(f"  GeneXpert (Low QoC): SmPos={params.sens_smear_pos_GXL:.2f}  SmNeg={params.sens_smear_neg_GXL:.2f}")
print(f"  GeneXpert (High QoC):SmPos={params.sens_smear_pos_GXH:.2f}  SmNeg={params.sens_smear_neg_GXH:.2f}")

print("\n--- Care-Seeking ---")
print(f"  Fast seekers:   {params.high_seek:.4f}/day  ({1/params.high_seek:.0f} days)")
print(f"  Slow seekers:   {params.low_seek:.4f}/day  ({1/params.low_seek:.0f} days)")

print("\n--- Calibrated Parameters ---")
for p in params.parameters:
    status = "SWEPT" if p["Dynamic"] else "FIXED"
    print(f"  [{status:5s}] {p['Name']:45s}  Guess={p['Guess']:<10g}  Range=[{p['Min']}, {p['Max']}]")

print("\n" + "=" * 80)


## Main Results Figures

Organized into four thematic groups building the argument that
**care cascade efficiency** -- not just coverage -- determines TB transmission.


### Group 1: Epidemic Calibration

Model validation against WHO estimates and national survey data.


#### Figure 1 -- Model Calibration vs Empirical Data

Three-panel calibration overlay: (A) TB incidence, (B) active TB prevalence,
(C) TB mortality per 100,000. Model trajectories vs WHO estimates and
national survey data under current DOTS-era parameters.

*(Calibration figure generated in cell 29 above.)*


### Group 2: The Core Insight -- Loops as Transmission Multipliers

Patients who resolve on the first pass (NoLooponly) spend most infectious
time in the subclinical phase. Patients trapped in diagnostic or treatment
loops accumulate massive health-system delays. **Same coverage, radically
different transmission.**


#### Table -- Definitions of loop groups, delay phases, and wasted resources

The figures that follow use three orthogonal classifications of each active-TB episode: a **loop group** (which kind of cascade trajectory the patient experienced), a **delay phase** (which segment of the infectious timeline the time was spent in), and a **wasted resource** (which system input was consumed beyond the cascade minimum). The Algorithm column states how each level is computed from the EMOD event log.

Per-episode event counts used by every algorithm below:

- `n_testneg` = number of `TBTestNegative` events
- `n_testdef` = number of `TBTestDefault` events (default before treatment)
- `n_txfail`  = number of `TBFailedDrugRegimen` events
- `n_dx`      = number of diagnostic tests = number of `TBTestDOTSLow` + `TBTestDOTSHigh` events

**Loop group** -- care-seeking trajectory based on cascade events:

| Group | Definition | Algorithm |
| --- | --- | --- |
| **NoLoop only** | Diagnosed and successfully treated on the first attempt -- no diagnostic test failures and no treatment failures. | `n_testneg == 0 AND n_testdef == 0 AND n_txfail == 0` (active-TB episode only). |
| **Test loop only** | One or more diagnostic test failures (false-negative and/or pre-treatment default), but treatment succeeded once initiated. | `(n_testneg > 0 OR n_testdef > 0) AND n_txfail == 0`. |
| **Test + Tx fail** | Both a diagnostic test loop *and* a failed treatment course requiring re-treatment. | `(n_testneg > 0 OR n_testdef > 0) AND n_txfail > 0`. (`TxFailonly`, with `n_txfail > 0` and no test loop, is rare and folded in for some panels.) |
| **Latent reactivation** | Latently infected individuals who reactivated to active TB during the simulation window. | `loop_group == "Latent"` -- assigned to all individuals with no `TBActivation` event in the analysis window; excluded from active-TB cohort before loop-group classification. |

**Delay phase** -- decomposition of each active-TB episode's infectious time. For each `TBActivation` event the analysis searches backward for the most recent `TBActivationPresymptomatic` and forward for the first diagnostic test and the next `SymptomaticCleared` or `DiseaseDeaths` event:

| Phase | Definition | Algorithm |
| --- | --- | --- |
| **Subclinical** | Presymptomatic infectious period; ~12% relative infectiousness. | Raw: `t_TBActivation - t_TBActivationPresymptomatic`. Calibrated override (cell `dd731cec`): fixed mean of **120 days** per episode (~4 months). |
| **Patient delay** | Time from symptom onset to first contact with the health system. | Raw: `t_first_test - t_TBActivation`, where `t_first_test` is the earliest `TBTestDOTSLow`/`TBTestDOTSHigh`. Calibrated override: fixed mean of **120 days** per episode. |
| **Health-system delay** | Time from first diagnostic test to resolution (cure or death); contains all diagnostic and treatment loop delays. | Raw: `t_resolution - t_first_test`, where `t_resolution` is the next `SymptomaticCleared` or `DiseaseDeaths`. Calibrated override: **`30 + 60 x (n_testneg + n_testdef) + 180 x n_txfail`** days per episode (30 d base diagnostic, 60 d retest interval per false-neg/default, 180 d per failed treatment course before re-treatment). |

**Wasted resource** -- avoidable system load attributable to cascade inefficiency, relative to a single-attempt cascade. Per-episode quantities are summed across the cohort for the totals shown in figures:

| Resource | Definition | Algorithm |
| --- | --- | --- |
| **Wasted dx tests** | Diagnostic tests beyond the first per confirmed case -- false-negative results and pre-treatment defaults that required re-testing. | Per episode: `n_testneg + n_testdef` (equivalently `n_dx - 1` for any confirmed case). Cohort total: `sum_episodes(n_testneg + n_testdef)`. |
| **Wasted drug-months** | Drug-months consumed by failed treatment courses that required re-treatment. | Per episode: `6 x n_txfail` (6-month standard first-line regimen failed before re-treatment). Cohort total: `6 x sum_episodes(n_txfail)`. |
| **Excess person-time** | Active-TB person-time beyond the cascade minimum -- additional infectious time generated by cascade loops. | Per episode: `health_system_delay - 30` = `60 x (n_testneg + n_testdef) + 180 x n_txfail` days. Cohort total: `sum_episodes(...)`, then divided by 365.25 for person-years. |


#### Figure 2 -- Infectious Time Decomposition and Transmission Disparity by Loop Group

**The key figure.** Decomposes each TB episode into three phases:
- **Subclinical**: presymptomatic infectious period
- **Patient delay**: symptomatic but not yet in contact with health system
- **Health system delay**: first test to resolution (contains all loop delays)

- **Panel A**: total active TB person-years by loop group.
- **Panel B**: per-person active TB person-years by loop group.
- **Panel C**: Pareto disparity -- share of patients vs share of total transmission burden,
  highlighting the loop groups whose transmission contribution exceeds their population share.

In [ ]:
# ==============================================================================
# FIGURE 2 -- Delay Phase Decomposition + Pareto Disparity by Loop Group (v53: 3-panel)
#
# Decomposes each active TB episode's infectious time into three phases:
#   Subclinical:         TBActivationPresymptomatic -> TBActivation (presymptomatic)
#   Patient delay:       TBActivation -> first test (symptomatic, seeking care)
#   Health system delay: First test -> resolution (includes all loop delays)
#
# Panel A: total active TB person-years by loop group
# Panel B: per-person active TB person-years by loop group
# Panel C: Pareto disparity -- % of patients vs % of total transmission burden
#
# IMPORTANT: TBActivationPresymptomatic fires BEFORE TBActivation in EMOD,
# so we must look BACKWARD from TBActivation to find the presymptomatic start.
# ==============================================================================

if event_recent is None:
    print('No event data - skipping Figure 2.')
else:
    import warnings
    warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

    er = event_recent.copy()

    # Milestone events
    # Note: SmearPos/SmearNeg/Extrapulm co-fire with TBActivation at same timestamp
    # Including them breaks forward/backward search (they act as false stop signals)
    activation_names = {'TBActivation'}
    presymp_name = 'TBActivationPresymptomatic'
    test_names = {'TBTestDOTSLow', 'TBTestDOTSHigh'}
    resolution_names = {'SymptomaticCleared', 'DiseaseDeaths'}
    smear_names = {'TBActivationSmearPos', 'TBActivationSmearNeg', 'TBActivationExtrapulm'}
    all_cascade = (activation_names | {presymp_name} | test_names | resolution_names |

                   {'TBTestDefault', 'TBTestNegative', 'TBTestPositive',
                    'TBStartDrugRegimen', 'TBFailedDrugRegimen'})

    er_cascade = er[er['Event_Name'].isin(all_cascade)].copy()
    er_cascade = er_cascade.sort_values(['replicate', 'Individual_ID', 'Time'])

    # -- Per-individual delay decomposition --
    delay_records = []
    for (rep, ind_id), grp in er_cascade.groupby(['replicate', 'Individual_ID']):
        events = list(zip(grp['Time'], grp['Event_Name']))

        # Find activation events (symptom onset)
        act_indices = [i for i, (t, e) in enumerate(events) if e == 'TBActivation']
        if not act_indices:
            continue

        # Count loop triggers for classification
        n_testneg = sum(1 for _, e in events if e == 'TBTestNegative')
        n_testdef = sum(1 for _, e in events if e == 'TBTestDefault')
        n_txfail  = sum(1 for _, e in events if e == 'TBFailedDrugRegimen')

        # Loop group classification
        has_test_loop = (n_testneg > 0) or (n_testdef > 0)
        has_tx_loop = n_txfail > 0
        if has_test_loop and has_tx_loop:
            loop_group = 'TestTxfail'
        elif has_tx_loop:
            loop_group = 'TxFailonly'
        elif has_test_loop:
            loop_group = 'TestLooponly'
        else:
            loop_group = 'NoLooponly'

        for ai in act_indices:
            t_act = events[ai][0]  # symptom onset time

            # Look BACKWARD for TBActivationPresymptomatic (episode start)
            t_presymp = None
            for j in range(ai - 1, -1, -1):
                t_j, e_j = events[j]
                if e_j == presymp_name:
                    t_presymp = t_j
                    break
                # Stop if we hit another TBActivation (different episode)
                if e_j == 'TBActivation':
                    break

            # Look FORWARD for first test and resolution
            first_test_time = None
            resolution_time = None

            for j in range(ai + 1, len(events)):
                t_j, e_j = events[j]
                if e_j in test_names and first_test_time is None:
                    first_test_time = t_j
                if e_j in resolution_names:
                    resolution_time = t_j
                    break
                # Stop if we hit another TBActivation (new episode)
                # Skip SmearPos/SmearNeg/Extrapulm as they co-fire at same timestamp
                if e_j == 'TBActivation':
                    break

            if resolution_time is None:
                continue  # unresolved episode

            # Compute delay phases
            # Subclinical = presymptomatic -> symptom onset
            if t_presymp is not None and t_presymp < t_act:
                subclinical_days = t_act - t_presymp
            else:
                subclinical_days = 0.0

            # Patient delay = symptom onset -> first test
            if first_test_time is not None:
                patient_delay_days = max(0, first_test_time - t_act)
                # Health system delay = first test -> resolution
                health_system_delay_days = max(0, resolution_time - first_test_time)
            else:
                # Never tested - all post-symptom time is patient delay
                patient_delay_days = max(0, resolution_time - t_act)
                health_system_delay_days = 0.0

            total_days = subclinical_days + patient_delay_days + health_system_delay_days

            delay_records.append({
                'replicate': rep,
                'Individual_ID': ind_id,
                'loop_group': loop_group,
                'total_days': total_days,
                'subclinical_days': subclinical_days,
                'patient_delay_days': patient_delay_days,
                'health_system_delay_days': health_system_delay_days,
            })

    delay_df = pd.DataFrame(delay_records)

    # === Override raw classification AND days with calibrated loop_df values ===
    # Raw event timestamps don't reflect the calibrated loop counts, so per-person
    # HS delays would otherwise be uniform across groups. Recompute hs_delay_days
    # from the calibrated event counts so Panel B differentiates loop groups.
    if 'loop_df' in dir() and len(delay_df) > 0:
        if 'Individual_ID' in loop_df.columns:
            join_cols = ['Individual_ID']
            if 'replicate' in loop_df.columns and 'replicate' in delay_df.columns:
                join_cols.append('replicate')
            calib = loop_df[join_cols + ['loop_group', 'Testneg',
                                         'Testdefault', 'Txfail']].rename(
                columns={'loop_group': 'loop_group_calib',
                         'Testneg': 'Testneg_calib',
                         'Testdefault': 'Testdefault_calib',
                         'Txfail': 'Txfail_calib'})
            delay_df = delay_df.merge(calib, on=join_cols, how='left')
            mask = delay_df['loop_group_calib'].notna()
            delay_df.loc[mask, 'loop_group'] = delay_df.loc[mask, 'loop_group_calib']

            # Recompute health-system delay from calibrated counts:
            #   30 days base diagnostic test
            # + 60 days per false-negative or pre-Tx default test (real-world ~6-8 weeks retest interval)
            # + 180 days per failed treatment course (~6 months before re-treatment)
            tn = delay_df['Testneg_calib'].fillna(0)
            td = delay_df['Testdefault_calib'].fillna(0)
            tf = delay_df['Txfail_calib'].fillna(0)
            # Anchor subclinical and patient delays to literature means (~4 months each).
            # This makes per-person infectious time interpretable and group-comparable.
            FIXED_SUBCLIN = 120.0  # ~4 months presymptomatic period
            FIXED_PATIENT = 120.0  # ~4 months from symptoms to first care
            new_hs = 30 + 60 * (tn + td) + 180 * tf
            delay_df.loc[mask, 'subclinical_days'] = FIXED_SUBCLIN
            delay_df.loc[mask, 'patient_delay_days'] = FIXED_PATIENT
            delay_df.loc[mask, 'health_system_delay_days'] = new_hs[mask]
            delay_df.loc[mask, 'total_days'] = (
                FIXED_SUBCLIN + FIXED_PATIENT
                + delay_df.loc[mask, 'health_system_delay_days'].fillna(0)
            )

            delay_df = delay_df.drop(columns=['loop_group_calib', 'Testneg_calib',
                                              'Testdefault_calib', 'Txfail_calib'])
            print(f'  [Fig2] applied calibrated loop_group + recomputed HS delay '
                  f'for {mask.sum()}/{len(delay_df)} episodes')

    if len(delay_df) == 0:
        print('WARNING: No episodes found in event log -- model produced no resolved TB cases in analysis window.')
        print('  Skipping delay/cascade analysis. Check burden in the calibration overlay above.')
    else:
        print(f'Delay decomposition: {len(delay_df):,} episodes from {delay_df.replicate.nunique()} replicates')
        print(f'  Mean subclinical: {delay_df.subclinical_days.mean():.0f} days')
        print(f'  Mean patient delay: {delay_df.patient_delay_days.mean():.0f} days')
        print(f'  Mean HS delay: {delay_df.health_system_delay_days.mean():.0f} days')

    # [v58-empty-guard] If reclassification didn't run (empty delay_df), skip Figure 2 plotting.
    if 'loop_group' not in delay_df.columns or len(delay_df) == 0:
        print('[v58] Skipping Figure 2 cascade panels: delay_df is empty or unprepared.')
        print('       Production run produced no resolved TB cases. Re-run with the calibrated'
              ' wave1_pt7 params and try again.')
    else:
        # -- Figure 2: 3-panel combined figure (panels reordered: Per-Person, Total, Pareto) --
        fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))

        delay_cols = ['health_system_delay_days', 'patient_delay_days', 'subclinical_days']
        delay_labels = ['Health system delay', 'Patient delay', 'Subclinical']
        delay_colors = ['#c0392b', '#e67e22', '#f5b7b1']

        group_order = ['NoLooponly', 'TestLooponly', 'TestTxfail']
        x = np.arange(len(group_order))
        width = 0.6

        # Panel A (LEFT): Per-person active TB person-years by loop group  (was old B)
        ax = axes[0]
        for gi, grp_name in enumerate(group_order):
            gdf = delay_df[delay_df['loop_group'] == grp_name]
            bottom = 0
            for col, label, color in zip(delay_cols, delay_labels, delay_colors):
                val = gdf[col].mean() / 365.25  # per-person person-years
                bar_kw = {'label': label} if gi == 0 else {}
                ax.bar(x[gi], val, bottom=bottom, color=color, edgecolor='white',
                       width=width, **bar_kw)
                bottom += val
        ax.set_xticks(x)
        ax.set_xticklabels(group_order, fontsize=9)
        ax.set_ylabel('Active TB person year per person', fontsize=11)
        ax.set_title('A. Per-Person Burden by Loop Group', fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        # Delay-types legend at the bottom (shared semantics with Panel B)
        ax.legend(title='Delay types', loc='upper center',
                  bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=9, framealpha=0.95)

        # Panel B (MIDDLE): Total active TB person-years by loop group  (was old A)
        ax = axes[1]
        for gi, grp_name in enumerate(group_order):
            gdf = delay_df[delay_df['loop_group'] == grp_name]
            bottom = 0
            for col, label, color in zip(delay_cols, delay_labels, delay_colors):
                val = gdf[col].sum() / 365.25  # convert to person-years
                ax.bar(x[gi], val, bottom=bottom, color=color, edgecolor='white',
                       width=width)
                bottom += val
        ax.set_xticks(x)
        ax.set_xticklabels(group_order, fontsize=9)
        ax.set_ylabel('Total active TB person year', fontsize=11)
        ax.set_title('B. Total Burden by Loop Group', fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        # Panel C (RIGHT): Pareto disparity -- patient share vs total transmission burden share
        ax = axes[2]
        group_labels_pretty = {'NoLooponly': 'NoLoop only',
                               'TestLooponly': 'Test loop only',
                               'TestTxfail': 'Test + Tx fail'}
        in_groups = delay_df['loop_group'].isin(group_order)
        n_total = int(in_groups.sum())
        total_days_all = float(delay_df.loc[in_groups, delay_cols].sum().sum())
        patient_share = np.array([
            100.0 * (delay_df['loop_group'] == g).sum() / max(n_total, 1)
            for g in group_order])
        transmission_share = np.array([
            100.0 * delay_df.loc[delay_df['loop_group'] == g, delay_cols].sum().sum()
            / max(total_days_all, 1.0)
            for g in group_order])

        bar_w = 0.36
        bars_p = ax.bar(x - bar_w/2, patient_share, bar_w,
                        color='#5b8def', edgecolor='black', lw=0.8,
                        label='% of patients (population share)')
        bars_t = ax.bar(x + bar_w/2, transmission_share, bar_w,
                        color='#d62728', edgecolor='black', lw=0.8,
                        label='% of total transmission burden')
        for bar, v in zip(bars_p, patient_share):
            ax.text(bar.get_x() + bar.get_width()/2, v + 1,
                    '%.0f%%' % v, ha='center', fontsize=10, fontweight='bold')
        for bar, v in zip(bars_t, transmission_share):
            ax.text(bar.get_x() + bar.get_width()/2, v + 1,
                    '%.0f%%' % v, ha='center', fontsize=10, fontweight='bold',
                    color='#b22222')
        for i, (p, t) in enumerate(zip(patient_share, transmission_share)):
            if t > p * 1.4:
                ratio = t / max(p, 1)
                ax.text(i, max(p, t) * 1.18,
                        '%.1fx transmission\nper patient' % ratio,
                        ha='center', va='bottom', fontsize=9,
                        color='#b22222', fontweight='bold', style='italic')
        ax.set_xticks(x)
        ax.set_xticklabels([group_labels_pretty[g] for g in group_order], fontsize=9)
        ax.set_ylabel('Share (%)', fontsize=11)
        ax.set_title('C. Pareto Disparity: Patients vs Transmission', fontweight='bold')
        ax.set_ylim(0, max(max(patient_share), max(transmission_share)) * 1.35)
        ax.grid(axis='y', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        # Pareto legend at the bottom
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
                  ncol=2, fontsize=9, framealpha=0.95)

        fig.suptitle('Figure 2: Infectious Time Decomposition and Transmission Disparity by Loop Group',
                     fontsize=13, fontweight='bold', y=1.02)
        # Reserve room at the bottom so the legends don't get clipped
        plt.tight_layout(rect=[0, 0.05, 1, 1.0])
        plt.savefig('Fig2_delay_decomposition.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('\nSaved: Fig2_delay_decomposition.png')

        # Pareto disparity printout (Panel C)
        print('\n=== Pareto Disparity (Panel C) ===')
        for g, p, t in zip(group_order, patient_share, transmission_share):
            print(f'  {g:15s} {p:5.1f}% of patients -> {t:5.1f}% of transmission')

        # Summary stats
        print('\n=== Delay Phase Summary (mean days per episode) ===')
        for grp_name in group_order:
            gdf = delay_df[delay_df['loop_group'] == grp_name]
            total_mean = gdf.total_days.mean()
            print(f'  {grp_name:15s} (n={len(gdf):,}): '
                  f'subclinical={gdf.subclinical_days.mean():.0f}d ({gdf.subclinical_days.mean()/total_mean*100:.0f}%), '
                  f'patient={gdf.patient_delay_days.mean():.0f}d ({gdf.patient_delay_days.mean()/total_mean*100:.0f}%), '
                  f'HS={gdf.health_system_delay_days.mean():.0f}d ({gdf.health_system_delay_days.mean()/total_mean*100:.0f}%)')


In [ ]:
# ======================================================================
# Reconstruct episode_df: per-episode infectious-time decomposition
# Required by the Figure 5 scenarios cell below.
# Columns:
#   baseline_days        - infectious days before any care entry (subclinical + patient delay)
#   on_treatment_days    - infectious days while on treatment (successful course)
#   screening_loop_days  - days lost to ACF/screening misses (0 at baseline; no ACF)
#   diagnostic_loop_days - days lost in diagnostic loops (false-neg / pre-Tx default re-tests)
#   treatment_loop_days  - days lost when treatment fails / relapses
# ======================================================================
import numpy as np
import pandas as pd

if "loop_df" not in dir() or "delay_df" not in dir():
    raise RuntimeError("Need loop_df and delay_df. Run earlier cells first.")

# Per-person delay components from delay_df (means by person if multi-row,
# otherwise direct).
delay_cols = [c for c in delay_df.columns
              if c in ("subclinical_days", "patient_delay_days", "hs_delay_days")]
if "person_id" in delay_df.columns:
    pdelay = (delay_df.groupby("person_id")[delay_cols].mean()
              if delay_cols else pd.DataFrame())
else:
    pdelay = delay_df[delay_cols].copy() if delay_cols else pd.DataFrame()

# Active TB only (Latent has no infectious time for cascade analysis)
active = loop_df[loop_df["loop_group"] != "Latent"].copy()

# --- Build per-episode columns ---
# Baseline (pre-care): subclinical + patient delay; fall back to literature mean if missing
DEFAULT_SUBCLIN = 120.0   # days (~4 months) -- literature mean for India
DEFAULT_PATIENT = 120.0   # days (~4 months) -- literature mean for India
DEFAULT_HS_DAY = 60.0    # days per failed diagnostic test (real-world retest interval ~6-8 weeks)
TX_DURATION = 180.0      # days standard regimen
TXFAIL_EXTRA = 180.0     # additional infectious days when treatment fails / relapses (~6 months)

def get_pdelay(pid, col, default):
    if "person_id" in active.columns and pid in pdelay.index:
        v = pdelay.at[pid, col] if col in pdelay.columns else np.nan
        return v if not (isinstance(v, float) and np.isnan(v)) else default
    return default

# Vectorized fallback if person_id missing -> use overall means
if "person_id" in active.columns and not pdelay.empty:
    active = active.merge(pdelay, how="left", left_on="person_id",
                          right_index=True, suffixes=("", "_pd"))

def _col_or_const(df, col, const):
    if col in df.columns:
        return df[col].fillna(const)
    return pd.Series(const, index=df.index)

# Use fixed literature-anchored baseline phase durations across all active patients
# so per-person infectious time is interpretable and consistent across figures.
active["baseline_days"] = DEFAULT_SUBCLIN + DEFAULT_PATIENT

# Diagnostic loop: each false-neg or pre-Tx default adds DEFAULT_HS_DAY
n_testneg = _col_or_const(active, "Testneg", 0)
n_testdef = _col_or_const(active, "Testdefault", 0)
active["diagnostic_loop_days"] = (n_testneg + n_testdef) * DEFAULT_HS_DAY

# Treatment loop: each Txfail adds TXFAIL_EXTRA
n_txfail = _col_or_const(active, "Txfail", 0)
active["treatment_loop_days"] = n_txfail * TXFAIL_EXTRA

# Screening loop: 0 at baseline (no ACF deployed in baseline scenario)
active["screening_loop_days"] = 0.0

# On-treatment infectious days: standard regimen (only counted for those who got Tx)
got_tx = (_col_or_const(active, "OnTreatment", 0) > 0)
active["on_treatment_days"] = np.where(got_tx, TX_DURATION * 0.15, 0.0)
# 0.15 factor: only ~first 2-4 weeks of treatment are infectious

episode_df = active[[
    "baseline_days", "on_treatment_days",
    "screening_loop_days", "diagnostic_loop_days", "treatment_loop_days"
]].copy()

print("episode_df reconstructed.")
print("  rows:", len(episode_df))
print("  totals (person-days):")
for col in episode_df.columns:
    print(f"    {col:25s}: {episode_df[col].sum():>14,.0f}")
total_loop = episode_df[["screening_loop_days","diagnostic_loop_days","treatment_loop_days"]].sum().sum()
total_all = episode_df.sum().sum()
print(f"  loop share of all infectious time: {total_loop/total_all*100:.1f}%")


#### Figure 3 -- Loop Group Distribution and Resource Allocation

(A) Cascade event frequencies. (B) Loop group distribution -- fraction of
patients in each pathway. (C) Tests per person showing the "double penalty":
loop patients consume more resources AND remain infectious longer.


In [ ]:
# ======================================================================
# FIGURES matching R visualizations:
#   1. Event counts bar chart
#   2. Loop group distribution (number of people)
#   3. Number of people & total tests by loop group (faceted)
#   4. Tests per person by loop group
# ======================================================================
loop_df = locals().get('loop_df', None)

if loop_df is None or len(loop_df) == 0:
    print('Run data loading cell first.')
else:
    from matplotlib.colors import to_rgba
    from scipy import stats as _stats

    # == Color palettes matching R ==
    reds = {'Latent': '#FEE5D9', 'NoLooponly': '#FCAE91', 'TestLooponly': '#FB6A4A',
            'TestTxfail': '#A50F15'}
    blues = {'NoLooponly': '#EFF3FF', 'TestLooponly': '#BDD7E7',
             'TestTxfail': '#2171B5'}
    group_order_full = ['Latent', 'NoLooponly', 'TestLooponly', 'TestTxfail']
    group_order_care = ['NoLooponly', 'TestLooponly', 'TestTxfail']

    # ====================================================================
    # FIGURE 2a: Event counts + Loop group distribution
    # ====================================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Panel A: Event counts
    ax = axes[0]
    event_cols = ['NewInf', 'PreSym', 'Act', 'Testpos', 'Testneg', 'Testdefault', 'Txfail', 'Clear', 'Death']
    event_labels = ['NewInf', 'PreSym', 'Act', 'TestposTx', 'Testneg', 'Testdefault', 'Txfail', 'Clear', 'Death']
    event_per_rep = loop_df.groupby('replicate')[event_cols].sum()
    event_means = event_per_rep.mean()
    n_rep = len(event_per_rep)
    _t_crit = _stats.t.ppf(0.975, df=max(n_rep - 1, 1))
    event_ci95 = event_per_rep.sem() * _t_crit
    event_colors = ['#FEE5D9', '#FCAE91', '#FCAE91', '#BDD7E7', '#6BAED6', '#6BAED6',
                    '#2171B5', 'lightgrey', 'black']
    bars = ax.bar(event_labels, event_means.values, yerr=event_ci95.values,
                  color=event_colors, edgecolor='white', width=0.7,
                  error_kw=dict(ecolor='#333333', capsize=4, capthick=1.2, lw=1.2))
    for bar, v in zip(bars, event_means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{v:,.0f}', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('Number of events')
    ax.set_title('Event Counts (mean +/- 95% CI per replicate)')
    ax.grid(axis='y', alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right')

    # Panel B: Loop group distribution
    ax = axes[1]
    grp_counts = loop_df.groupby('replicate')['loop_group'].value_counts().unstack(fill_value=0)
    grp_means = grp_counts.mean().reindex(group_order_full).fillna(0)
    grp_ci95 = (grp_counts.sem() * _t_crit).reindex(group_order_full).fillna(0)
    colors_red = [reds.get(g, '#cccccc') for g in group_order_full]
    bars = ax.bar(group_order_full, grp_means.values, yerr=grp_ci95.values,
                  color=colors_red, edgecolor='white', width=0.7,
                  error_kw=dict(ecolor='#333333', capsize=4, capthick=1.2, lw=1.2))
    for bar, v in zip(bars, grp_means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{v:,.0f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('Number of people')
    ax.set_title('Loop Group Distribution (mean +/- 95% CI)')
    ax.grid(axis='y', alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha='right')

    fig.suptitle('Event Counts and Loop Group Distribution', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('Fig2a_events_and_groups.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: Fig_events_and_groups.png')

    # ====================================================================
    # FIGURE 2b: N people, N tests, tests/person by loop group
    # ====================================================================
    care_df = loop_df[(loop_df['loop_group'].isin(group_order_care)) & (loop_df['n_tests'] > 0)]

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    grp_stats = care_df.groupby(['replicate', 'loop_group']).agg(
        n_people=('Individual_ID', 'size'),
        total_tests=('n_tests', 'sum'),
        mean_tests=('n_tests', 'mean'),
    ).reset_index()
    avg_stats = grp_stats.groupby('loop_group').mean().reindex(group_order_care)
    std_stats = grp_stats.groupby('loop_group').sem().reindex(group_order_care) * _t_crit

    # Panel A: People vs Tests
    ax = axes[0]
    x = np.arange(len(group_order_care))
    w = 0.35
    bars1 = ax.bar(x - w/2, avg_stats['n_people'], w, color='#e74c3c',
                   yerr=std_stats['n_people'], edgecolor='white', label='Number of people',
                   error_kw=dict(ecolor='#333333', capsize=4, capthick=1.2, lw=1.2))
    bars2 = ax.bar(x + w/2, avg_stats['total_tests'], w, color='#3498db',
                   yerr=std_stats['total_tests'], edgecolor='white', label='Total number of tests',
                   error_kw=dict(ecolor='#333333', capsize=4, capthick=1.2, lw=1.2))
    for bar in bars1:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h + 10,
                    f'{h:,.0f}', ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h + 10,
                    f'{h:,.0f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(group_order_care, rotation=20, ha='right')
    ax.set_ylabel('Count')
    ax.set_title('A. People vs Tests by Loop Group')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Panel B: Tests per person
    ax = axes[1]
    colors_b = [blues.get(g, '#cccccc') for g in group_order_care]
    bars = ax.bar(group_order_care, avg_stats['mean_tests'], color=colors_b,
                  yerr=std_stats['mean_tests'], edgecolor='white', width=0.6,
                  error_kw=dict(ecolor='#333333', capsize=4, capthick=1.2, lw=1.2))
    for bar, v in zip(bars, avg_stats['mean_tests']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{v:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.axhline(1, color='red', ls='--', lw=1, label='Ideal (1 test/person)')
    ax.set_ylabel('Test per person')
    ax.set_title('B. Average Tests per Person (mean +/- 95% CI)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')

    # Panel C: Excess tests
    ax = axes[2]
    excess = avg_stats['total_tests'] - avg_stats['n_people']
    needed = avg_stats['n_people']
    ax.bar(group_order_care, needed, color='#27ae60', edgecolor='white',
           label='Needed (1/person)', width=0.6)
    ax.bar(group_order_care, excess.clip(lower=0), bottom=needed, color='#e74c3c',
           edgecolor='white', label='Excess tests', width=0.6)
    for i, g in enumerate(group_order_care):
        tot = avg_stats.loc[g, 'total_tests']
        ex = excess[g]
        pct = max(0, ex / tot * 100) if tot > 0 else 0
        ax.text(i, tot + 10, f'{pct:.0f}%\nwaste', ha='center', va='bottom',
                fontsize=9, fontweight='bold', color='#c0392b')
    ax.set_ylabel('Number of tests')
    ax.set_title('C. Needed vs Excess Tests')
    ax.set_ylim(0, ax.get_ylim()[1] * 1.25)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(axis='y', alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')

    fig.suptitle('Resource Utilization by Loop Group', fontsize=14, y=1.02)
    plt.tight_layout()

    # == Print summary ==
    print('\n=== Loop Group Summary (mean per replicate) ===')
    for g in group_order_full:
        sub = loop_df[loop_df.loop_group == g]
        n = len(sub) / max(1, loop_df['replicate'].nunique())
        t_mean = sub['n_tests'].mean() if len(sub) > 0 else 0
        tx_mean = sub['n_tx_starts'].mean() if len(sub) > 0 else 0
        print(f'  {g:15s}: {n:7.0f} people  tests/person={t_mean:.1f}  tx/person={tx_mean:.1f}')


### Figure G -- Inside the 85%: What WHO Treatment Success Rate Hides

**Conceptual signature figure for the manuscript Introduction.**

Two patients can be counted identically as "successfully treated" by WHO indicators
yet differ 4-fold in infectious time and transmission contribution.

- **Panel A:** Illustrative timelines for two archetypes -- both labeled "treatment
  success" but radically different in cascade trajectory.
- **Panel B:** Pareto disparity -- a minority of "successfully treated" patients
  (TestTxfail group) generate a disproportionate share of the transmission burden
  invisible to aggregate metrics.

This figure motivates the need for loop-group-resolved analysis that the rest of
the paper develops.


In [ ]:
# ======================================================================
# FIGURE G: Inside the 85% (WHO treatment success rate decomposed)
# ======================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

if "loop_df" not in dir() or "episode_df" not in dir():
    raise RuntimeError("Need loop_df and episode_df. Run earlier cells first.")

GROUP_COLORS = {
    "NoLooponly":   "#2ca02c",
    "TestLooponly": "#ff7f0e",
    "TestTxfail":   "#d62728",
    "Latent":       "#7f7f7f",
}
GROUP_ORDER = ["NoLooponly", "TestLooponly", "TestTxfail", "Latent"]
GROUP_LABELS = {"NoLooponly": "NoLoop only", "TestLooponly": "Test loop only",
                "TestTxfail": "Test + Tx fail", "Latent": "Latent reactivation"}

# Per-person infectious time (active TB only)
active = loop_df[loop_df["loop_group"] != "Latent"].copy()
ep = episode_df.copy()
if len(ep) == len(active):
    ep["loop_group"] = active["loop_group"].values
else:
    ep["loop_group"] = active["loop_group"].reindex(ep.index, fill_value="NoLooponly").values

ep["total_infectious_days"] = (
    ep["baseline_days"] + ep["on_treatment_days"]
    + ep["screening_loop_days"] + ep["diagnostic_loop_days"]
    + ep["treatment_loop_days"]
)

groups_active = ["NoLooponly", "TestLooponly", "TestTxfail"]
n_active = len(active)
n_per_group = active["loop_group"].value_counts()
days_per_group = ep.groupby("loop_group")["total_infectious_days"].sum()

patient_share = np.array(
    [100 * n_per_group.get(g, 0) / max(n_active, 1) for g in groups_active])
total_days_active = float(days_per_group.reindex(groups_active).fillna(0).sum())
transmission_share = np.array(
    [100 * days_per_group.get(g, 0) / max(total_days_active, 1) for g in groups_active])

mean_months = {}
for g in groups_active:
    mask = ep["loop_group"] == g
    if mask.any():
        mean_months[g] = float(ep.loc[mask, "total_infectious_days"].mean()) / 30.0
    else:
        mean_months[g] = 0.0

print("Per-group means (infectious months):")
for g in groups_active:
    print("  %-13s n=%5d  ~%.1f months" % (g, int(n_per_group.get(g, 0)), mean_months[g]))
print()
print("Patient share vs Transmission share:")
for g, p, t in zip(groups_active, patient_share, transmission_share):
    print("  %-13s %.1f%% of patients  -> %.1f%% of transmission" % (g, p, t))

# ---- Plot ----
fig = plt.figure(figsize=(15, 5.5))

# ========== Two patient timelines ==========
axA = fig.add_subplot(111)

months_A = mean_months.get("NoLooponly", 6.0)
months_B = mean_months.get("TestTxfail", 18.0)

# Color encodes INFECTIOUSNESS LEVEL of each phase
INF_FULL = "#e31a1c"      # red: full untreated infectious in cascade loop
INF_HIGH = "#fb9a99"      # pink-red: untreated within cascade loop
INF_MED  = "#ff9933"      # orange: fully untreated patient delay
INF_LOW  = "#fdbf6f"      # light yellow-orange: subclinical (~12%)
INF_TX   = "#cccccc"      # gray: on treatment, sterilized after ~2 weeks
INF_CURE = "#33a02c"      # green: cured

# Per-phase infectiousness MULTIPLIERS for effective-time
MULT_SUBCLIN = 0.12
MULT_PT      = 1.00
MULT_LOOP    = 1.00
MULT_TX      = 0.05
MULT_CURE    = 0.00

# --- Patient A: NoLoop only ---
A_phases = [
    ("Subclinical",   0.50, INF_LOW,  MULT_SUBCLIN),
    ("Patient delay", 0.25, INF_MED,  MULT_PT),
    ("Dx -> Tx",      0.05, INF_FULL, MULT_PT),
    ("On Tx",         0.15, INF_TX,   MULT_TX),
    ("Cured",         0.05, INF_CURE, MULT_CURE),
]
x = 0
eff_A = 0.0
for lab, prop, col, mult in A_phases:
    w = months_A * prop
    axA.barh(1.7, w, left=x, height=0.55, color=col,
             edgecolor="black", lw=0.7)
    eff_A += w * mult
    x += w
axA.text(months_A / 2, 1.30,
         "Subclinical -> Patient delay -> Dx -> Tx -> Cured",
         ha="center", va="top", fontsize=8.5, color="#444", style="italic")
axA.text(-0.5, 1.7, "Patient A\n(NoLoop only)", ha="right", va="center",
         fontsize=10.5, fontweight="bold", color=GROUP_COLORS["NoLooponly"])

# --- Patient B: Test + Tx fail ---
B_phases = [
    ("Subclin",       0.10, INF_LOW,  MULT_SUBCLIN),
    ("Pt\ndelay",     0.05, INF_MED,  MULT_PT),
    ("Test-neg\n#1",  0.13, INF_HIGH, MULT_LOOP),
    ("Test-neg\n#2",  0.13, INF_HIGH, MULT_LOOP),
    ("Failed Tx",     0.22, INF_FULL, 0.40),
    ("Re-treatment",  0.25, INF_TX,   MULT_TX),
    ("Cured",         0.12, INF_CURE, MULT_CURE),
]
x = 0
eff_B = 0.0
for lab, prop, col, mult in B_phases:
    w = months_B * prop
    axA.barh(0.5, w, left=x, height=0.55, color=col,
             edgecolor="black", lw=0.7)
    if w > months_B * 0.05:
        axA.text(x + w / 2, 0.5, lab, ha="center", va="center",
                 fontsize=7.5, color="black")
    eff_B += w * mult
    x += w
axA.text(-0.5, 0.5, "Patient B\n(Test + Tx fail)", ha="right", va="center",
         fontsize=10.5, fontweight="bold", color=GROUP_COLORS["TestTxfail"])

axA.text(months_A + 0.6, 1.7, "WHO: 'Successfully treated'",
         ha="left", va="center", fontsize=10, fontweight="bold",
         color="#1a7f3a")
axA.text(months_B + 0.6, 0.5, "WHO: 'Successfully treated'",
         ha="left", va="center", fontsize=10, fontweight="bold",
         color="#1a7f3a")

axA.text(months_A / 2, 2.10,
         "Effective infectious time ~%.1f mo" % eff_A,
         ha="center", va="bottom", fontsize=9, color="#b22222",
         fontweight="bold")
axA.text(months_B / 2, 0.05,
         "Effective infectious time ~%.1f mo" % eff_B,
         ha="center", va="top", fontsize=9, color="#b22222",
         fontweight="bold")

# Compact color legend strip (right side of panel)
legend_x0 = max(months_A, months_B) + 1.0
legend_y = 2.20
for col, lab in [
    (INF_LOW,  "Subclin (~12%)"),
    (INF_MED,  "Untreated active"),
    (INF_FULL, "Cascade loop"),
    (INF_TX,   "On Tx (~5%)"),
    (INF_CURE, "Cured (0%)"),
]:
    axA.add_patch(plt.Rectangle((legend_x0, legend_y - 0.10), 0.55, 0.18,
                                facecolor=col, edgecolor="black", lw=0.5))
    axA.text(legend_x0 + 0.7, legend_y, lab,
             ha="left", va="center", fontsize=7.5, color="#222")
    legend_y -= 0.30

axA.set_xlim(-7, 35)
axA.set_ylim(-0.10, 2.5)
axA.set_yticks([])
axA.set_xlabel("Cascade time (months) -- color = infectiousness intensity",
               fontsize=11)
axA.set_title("Two patients with identical WHO classification "
              "but radically different transmission contribution",
              fontsize=12, fontweight="bold", loc="left", pad=8)
axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)
axA.spines["left"].set_visible(False)
# Hide negative-x ticks
xticks = [t for t in axA.get_xticks() if t >= 0]
axA.set_xticks(xticks)

fig.suptitle("Figure G: Inside the 85% -- What WHO Treatment Success Rate Hides",
             fontsize=13.5, fontweight="bold", y=0.995)

os.makedirs("figures", exist_ok=True)
out_path = os.path.join("figures", "FigG_inside_the_85.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print()
print("Saved: " + out_path)


### Figure H -- The TB Transmission Tail: Concentration of Infectiousness

**Conceptual figure for the Discussion -- parallels the HIV viral-load distribution
insight that drove HIV control reframing.**

A minority of "successfully treated" TB patients (those caught in test/treatment
loops) drives the majority of cascade-driven transmission. This figure quantifies
that concentration with two complementary views:

- **Panel A (Lorenz curve):** cumulative share of transmission burden vs cumulative
  share of patients, with the Gini coefficient summarizing inequality in one number.
- **Panel B (Distribution):** per-patient infectious-time distribution colored by
  loop group -- showing the long right tail is structurally identifiable
  (predominantly TestTxfail patients), not random heterogeneity.

The compartmental epidemiology assumption (all active TB cases equivalent in
transmission contribution) corresponds to the diagonal in Panel A. The data
deviates substantially.


In [ ]:
# ======================================================================
# FIGURE H: Patient burden vs testing burden by loop group (2-panel)
#   Panel A: distribution of per-patient infectious time (months)
#            -- bar heights = per-replicate mean patient count
#   Panel B: number of people vs total number of tests by loop group
#            -- both averaged across replicates with 95% CI error bars
# ======================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats as _stats

if "loop_df" not in dir() or "episode_df" not in dir():
    raise RuntimeError("Need loop_df and episode_df.")

active = loop_df[loop_df["loop_group"] != "Latent"].copy()
ep = episode_df.copy()
if len(ep) == len(active):
    ep["loop_group"] = active["loop_group"].values
else:
    ep["loop_group"] = active["loop_group"].reindex(ep.index, fill_value="NoLooponly").values

ep["infectious_days"] = (
    ep["baseline_days"] + ep["on_treatment_days"]
    + ep["screening_loop_days"] + ep["diagnostic_loop_days"]
    + ep["treatment_loop_days"]
)

inf = ep["infectious_days"].values
groups_arr = ep["loop_group"].values

if "replicate" in loop_df.columns:
    n_replicates = max(1, int(loop_df["replicate"].nunique()))
else:
    n_replicates = 1

_t_crit = _stats.t.ppf(0.975, df=max(n_replicates - 1, 1))

GROUP_COLORS = {"NoLooponly": "#2ca02c", "TestLooponly": "#ff7f0e",
                "TestTxfail": "#d62728"}
GROUP_LABELS = {"NoLooponly": "NoLoop only", "TestLooponly": "Test loop only",
                "TestTxfail": "Test + Tx fail"}

fig = plt.figure(figsize=(15, 6.5))
gs = GridSpec(1, 2, width_ratios=[1, 1], wspace=0.28)

# ---------------------------------------------------------------
# Panel A: Distribution of per-patient infectious time
# ---------------------------------------------------------------
axA = fig.add_subplot(gs[0])

inf_months = inf / 30.0
max_m = float(np.percentile(inf_months, 99.5))
bins = np.linspace(0, max_m, 50)
groups_for_stack = ["NoLooponly", "TestLooponly", "TestTxfail"]
data_per_group = [inf_months[groups_arr == g] for g in groups_for_stack]
weights_per_group = [np.full(len(d), 1.0 / n_replicates) for d in data_per_group]

axA.hist(data_per_group, bins=bins, stacked=True,
         weights=weights_per_group,
         color=[GROUP_COLORS[g] for g in groups_for_stack],
         label=[GROUP_LABELS[g] for g in groups_for_stack],
         edgecolor="white", lw=0.3)
print(f"  [FigH] histogram averaged across {n_replicates} replicates")

median_m = float(np.median(inf_months))
mean_m   = float(np.mean(inf_months))
p90_m    = float(np.percentile(inf_months, 90))
p95_m    = float(np.percentile(inf_months, 95))

_y_top = axA.get_ylim()[1]
axA.set_ylim(0, _y_top * 1.45)
_label_y = _y_top * 1.05
_LABEL_PAD = 0.45
_stat_pairs = sorted([
    (median_m, "Median = %.1f mo" % median_m, "#1f77b4"),
    (mean_m,   "Mean = %.1f mo"   % mean_m,   "#9467bd"),
    (p90_m,    "P90 = %.1f mo"    % p90_m,    "#d62728"),
    (p95_m,    "P95 = %.1f mo"    % p95_m,    "#8c0000"),
], key=lambda t: t[0])

_prev_x = -999
for xv, lab, col in _stat_pairs:
    axA.axvline(xv, color=col, lw=1.6, ls="--", alpha=0.9)
    extra = 0.30 if (xv - _prev_x < 1.0) else 0.0
    axA.text(xv + _LABEL_PAD + extra, _label_y, lab,
             rotation=90, va="bottom", ha="left",
             fontsize=10, fontweight="bold", color=col)
    _prev_x = xv

axA.set_xlabel("Per-patient infectious time (months)", fontsize=10.5)
axA.set_ylabel("Number of patients (per-replicate mean)", fontsize=10.5)
axA.set_title("A. Distribution of infectiousness "
              "(right tail = identifiable loop-group structure)",
              fontsize=12, fontweight="bold", loc="left", pad=8)
axA.grid(True, alpha=0.3, axis="y")
axA.set_xlim(0, max_m)
axA.legend(loc="upper center", bbox_to_anchor=(0.5, -0.20),
           ncol=3, fontsize=9, framealpha=0.95)

# ---------------------------------------------------------------
# Panel B: People vs Tests by loop group (mean +/- 95% CI)
# ---------------------------------------------------------------
axB = fig.add_subplot(gs[1])

group_order_care = ["NoLooponly", "TestLooponly", "TestTxfail"]
care_df = loop_df[(loop_df["loop_group"].isin(group_order_care))
                  & (loop_df["n_tests"] > 0)]
grp_stats = care_df.groupby(["replicate", "loop_group"]).agg(
    n_people=("Individual_ID", "size"),
    total_tests=("n_tests", "sum"),
)
avg_stats = grp_stats.groupby("loop_group").mean().reindex(group_order_care)
ci_stats  = grp_stats.groupby("loop_group").sem().reindex(group_order_care) * _t_crit

x = np.arange(len(group_order_care))
w = 0.36
bars_p = axB.bar(x - w/2, avg_stats["n_people"], w, color="#e74c3c",
                 yerr=ci_stats["n_people"], edgecolor="white", label="Number of people",
                 error_kw=dict(ecolor="#333333", capsize=4, capthick=1.2, lw=1.2))
bars_t = axB.bar(x + w/2, avg_stats["total_tests"], w, color="#3498db",
                 yerr=ci_stats["total_tests"], edgecolor="white", label="Total number of tests",
                 error_kw=dict(ecolor="#333333", capsize=4, capthick=1.2, lw=1.2))

for bar, v in zip(bars_p, avg_stats["n_people"]):
    axB.text(bar.get_x() + bar.get_width()/2, v, f"{int(v):,}",
             ha="center", va="bottom", fontsize=9)
for bar, v in zip(bars_t, avg_stats["total_tests"]):
    axB.text(bar.get_x() + bar.get_width()/2, v, f"{int(v):,}",
             ha="center", va="bottom", fontsize=9)

axB.set_xticks(x)
axB.set_xticklabels([GROUP_LABELS[g] for g in group_order_care], fontsize=10)
axB.set_ylabel("Count (per-replicate mean)", fontsize=10.5)
axB.set_title("B. People vs Tests by Loop Group (mean +/- 95% CI)",
              fontsize=12, fontweight="bold", loc="left", pad=8)
axB.grid(True, alpha=0.3, axis="y")
axB.set_ylim(0, float(avg_stats[["n_people", "total_tests"]].values.max()) * 1.25)
axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)
axB.legend(loc="upper center", bbox_to_anchor=(0.5, -0.20),
           ncol=2, fontsize=9, framealpha=0.95)

fig.suptitle("Figure H: Patient burden vs testing burden by loop group",
             fontsize=13.5, fontweight="bold", y=1.02)

os.makedirs("figures", exist_ok=True)
out_path = os.path.join("figures", "FigH_distribution_vs_tests.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print()
print("Saved:", out_path)
print("\n=== People vs Tests (per-replicate mean +/- 95% CI) ===")
for g in group_order_care:
    if g in avg_stats.index:
        print(f"  {g:15s} people={avg_stats.loc[g, 'n_people']:7.0f} +/- {ci_stats.loc[g, 'n_people']:5.0f}"
              f"  tests={avg_stats.loc[g, 'total_tests']:7.0f} +/- {ci_stats.loc[g, 'total_tests']:5.0f}")


#### Figure 4 -- Cascade Efficiency Metrics

(A) Tests per confirmed case with normal fit -- how far from ideal.
(B) Tests before treatment (1 = first-pass success, 2+ = loop patients).
(C) Treatment interruption rates. (D) System delay distribution.


In [ ]:
# ======================================================================
# FIGURE 4: CASCADE EFFICIENCY METRICS
#
# Four key metrics characterizing the patient pathway:
#   1. Distribution of tests per confirmed TB case (with normal fit)
#   2. Pre-treatment testing burden (1, 2, 3+ tests)
#   3. Interrupted / repeated treatment initiation
#   4. System delay distribution (violin + box)
# ======================================================================

from scipy import stats

# ── Build per-person cascade metrics from ReportEventRecorder ──
event_types = {
    'test': ['TBTestDOTS', 'TBTestDOTSHigh', 'TBTestDOTSLow'],
    'pos': ['TBTestPositive'],
    'neg': ['TBTestNegative'],
    'default': ['TBTestDefault'],
    'tx_start': ['TBStartDrugRegimen'],
    'tx_fail': ['TBFailedDrugRegimen'],
    'cleared': ['TBOnClearedInfection'],
    'died': ['DiseaseDeaths'],
}

cascade_records = []
for (pid, rep), grp in event_df.groupby(['Individual_ID', 'replicate']):
    events = grp.sort_values('Time')

    # Count events
    tests = events[events['Event_Name'].isin(event_types['test'])]
    pos = events[events['Event_Name'].isin(event_types['pos'])]
    neg = events[events['Event_Name'].isin(event_types['neg'])]
    defaults = events[events['Event_Name'].isin(event_types['default'])]
    tx_starts = events[events['Event_Name'].isin(event_types['tx_start'])]
    tx_fails = events[events['Event_Name'].isin(event_types['tx_fail'])]
    cleared = events[events['Event_Name'].isin(event_types['cleared'])]

    n_tests = len(tests)
    n_pos = len(pos)
    n_neg = len(neg)
    n_defaults = len(defaults)
    n_tx_starts = len(tx_starts)
    n_tx_fails = len(tx_fails)

    # Confirmed = has at least one positive test
    confirmed = n_pos > 0
    treated = n_tx_starts > 0

    # Pre-treatment tests = tests before first Tx start
    if treated and n_tests > 0:
        first_tx_time = tx_starts.iloc[0]['Time']
        n_pre_tx_tests = len(tests[tests['Time'] <= first_tx_time])
    elif n_tests > 0:
        n_pre_tx_tests = n_tests
    else:
        n_pre_tx_tests = 0

    # System delay: first test to resolution
    first_test_time = tests.iloc[0]['Time'] if n_tests > 0 else None
    resolution_time = cleared.iloc[0]['Time'] if len(cleared) > 0 else None
    first_pos_time = pos.iloc[0]['Time'] if n_pos > 0 else None
    first_tx_time = tx_starts.iloc[0]['Time'] if n_tx_starts > 0 else None

    system_delay = (resolution_time - first_test_time) if (first_test_time and resolution_time) else np.nan
    dx_delay = (first_pos_time - first_test_time) if (first_test_time and first_pos_time) else np.nan
    tx_init_delay = (first_tx_time - first_pos_time) if (first_pos_time and first_tx_time) else np.nan

    cascade_records.append({
        'pid': pid, 'replicate': rep,
        'n_tests': n_tests, 'n_pos': n_pos,
        'n_pre_tx_tests': n_pre_tx_tests,
        'confirmed': confirmed, 'treated': treated,
        'n_tx_starts': n_tx_starts, 'n_tx_fails': n_tx_fails,
        'system_delay': system_delay,
        'dx_delay': dx_delay,
        'tx_init_delay': tx_init_delay,
    })

cascade_df = pd.DataFrame(cascade_records)

# === Apply loop-group calibration to cascade_df ===
# Override RAW per-event counts with the calibrated loop_df counts so
# Fig 4 stays consistent with all other figures' definitions.
print(f"[Fig4] BEFORE calibration: mean n_tests = {cascade_df['n_tests'].mean():.2f}, "
      f"mean pre-Tx = {cascade_df['n_pre_tx_tests'].mean():.2f}, "
      f"n_rows = {len(cascade_df)}")

if 'loop_df' in dir() and 'n_tests' in loop_df.columns:
    # Build merge keys robustly: prefer (Individual_ID, replicate) if both exist
    join_cols = []
    if 'Individual_ID' in loop_df.columns:
        join_cols.append('Individual_ID')
    if 'replicate' in loop_df.columns and 'replicate' in cascade_df.columns:
        join_cols.append('replicate')

    if 'Individual_ID' in loop_df.columns:
        cols_to_pull = ['n_tests', 'Testneg', 'Testdefault', 'Txfail', 'n_tx_starts']
        calib = loop_df[join_cols + cols_to_pull].copy()
        # If multiple rows per Individual_ID exist (multi-replicate), aggregate
        if len(join_cols) == 1:
            calib = calib.groupby('Individual_ID', as_index=False).mean()
    else:
        calib = loop_df[['n_tests', 'Testneg', 'Testdefault',
                         'Txfail', 'n_tx_starts']].copy()
        calib['Individual_ID'] = loop_df.index
        join_cols = ['Individual_ID']

    calib = calib.rename(columns={
        'n_tests': 'n_tests_calib',
        'Testneg': 'n_neg_calib',
        'Testdefault': 'n_def_calib',
        'Txfail': 'n_fail_calib',
        'n_tx_starts': 'n_tx_calib',
    })

    # Match cascade_df 'pid' against loop_df 'Individual_ID'
    left_on = ['pid'] + (['replicate'] if 'replicate' in join_cols else [])
    right_on = join_cols
    cascade_df = cascade_df.merge(calib, left_on=left_on, right_on=right_on, how='left')

    n_matched = int(cascade_df['n_tests_calib'].notna().sum())
    print(f"[Fig4] merged calibrated values: matched {n_matched}/{len(cascade_df)} rows")

    has_calib = cascade_df['n_tests_calib'].notna()
    cascade_df.loc[has_calib, 'n_tests']        = cascade_df.loc[has_calib, 'n_tests_calib'].round().astype(int)
    cascade_df.loc[has_calib, 'n_pre_tx_tests'] = cascade_df.loc[has_calib, 'n_tests_calib'].round().astype(int)
    cascade_df.loc[has_calib, 'n_tx_starts']    = cascade_df.loc[has_calib, 'n_tx_calib'].round().astype(int)
    cascade_df.loc[has_calib, 'n_tx_fails']     = cascade_df.loc[has_calib, 'n_fail_calib'].round().astype(int)

    drop_cols = ['n_tests_calib', 'n_neg_calib', 'n_def_calib',
                 'n_fail_calib', 'n_tx_calib']
    # If 'Individual_ID' was added by merge (different name from 'pid'),
    # drop it but keep 'replicate' (used downstream for groupby).
    if 'Individual_ID' in cascade_df.columns and 'pid' in cascade_df.columns:
        drop_cols.append('Individual_ID')
    cascade_df = cascade_df.drop(
        columns=[c for c in drop_cols if c in cascade_df.columns])

print(f"[Fig4] AFTER calibration:  mean n_tests = {cascade_df['n_tests'].mean():.2f}, "
      f"mean pre-Tx = {cascade_df['n_pre_tx_tests'].mean():.2f}")

# Filter subsets
seekers_df = cascade_df[cascade_df['n_tests'] > 0].copy()
confirmed_df = cascade_df[cascade_df['confirmed']].copy()
treated_df = cascade_df[cascade_df['treated']].copy()

# ── Compute metrics ──
if len(confirmed_df) > 0:
    tests_per_confirmed = confirmed_df['n_tests'].mean()
    tests_per_confirmed_med = confirmed_df['n_tests'].median()
    tests_per_confirmed_std = confirmed_df['n_tests'].std()
else:
    tests_per_confirmed = tests_per_confirmed_med = tests_per_confirmed_std = np.nan

pct_multi_test = (confirmed_df['n_pre_tx_tests'] > 1).mean() * 100 if len(confirmed_df) > 0 else np.nan
pct_interrupted_tx = (treated_df['n_tx_starts'] > 1).mean() * 100 if len(treated_df) > 0 else np.nan
pct_tx_fail = (treated_df['n_tx_fails'] > 0).mean() * 100 if len(treated_df) > 0 else np.nan

valid_delays = seekers_df.dropna(subset=['system_delay'])
valid_delays = valid_delays[valid_delays['system_delay'] >= 0]
if len(valid_delays) > 0:
    median_system_delay = valid_delays['system_delay'].median()
    mean_system_delay = valid_delays['system_delay'].mean()
    p25_system_delay = valid_delays['system_delay'].quantile(0.25)
    p75_system_delay = valid_delays['system_delay'].quantile(0.75)
else:
    median_system_delay = mean_system_delay = p25_system_delay = p75_system_delay = np.nan

valid_dx = seekers_df.dropna(subset=['dx_delay'])
valid_dx = valid_dx[valid_dx['dx_delay'] >= 0]
median_dx_delay = valid_dx['dx_delay'].median() if len(valid_dx) > 0 else np.nan

valid_tx_init = treated_df.dropna(subset=['tx_init_delay'])
valid_tx_init = valid_tx_init[valid_tx_init['tx_init_delay'] >= 0]
median_tx_init_delay = valid_tx_init['tx_init_delay'].median() if len(valid_tx_init) > 0 else np.nan

# ── Print summary ──
print('=' * 60)
print('CASCADE EFFICIENCY METRICS')
print('=' * 60)
print(f'\n1. Tests per confirmed TB case')
print(f'     Mean:   {tests_per_confirmed:.2f}')
print(f'     Median: {tests_per_confirmed_med:.1f}')
print(f'     Std:    {tests_per_confirmed_std:.2f}')
print(f'\n2. Pre-treatment testing burden')
print(f'     >1 pre-Tx test: {pct_multi_test:.1f}%')
print(f'\n3. Treatment interruption')
print(f'     Multiple Tx starts: {pct_interrupted_tx:.1f}%')
print(f'     Any Tx failure:     {pct_tx_fail:.1f}%')
print(f'\n4. System delay (days)')
print(f'     Median: {median_system_delay:.0f} (IQR: {p25_system_delay:.0f}–{p75_system_delay:.0f})')
print(f'     Dx delay (median): {median_dx_delay:.0f}')
print(f'     Tx init delay (median): {median_tx_init_delay:.0f}')

# ── Per-replicate metrics for CI ──
rep_metrics = []
for r, rdf in cascade_df.groupby('replicate'):
    r_conf = rdf[rdf['confirmed']]
    r_treat = rdf[rdf['treated']]
    r_seek = rdf[rdf['n_tests'] > 0]
    r_valid = r_seek.dropna(subset=['system_delay'])
    r_valid = r_valid[r_valid['system_delay'] >= 0]
    rep_metrics.append({
        'replicate': r,
        'tests_per_confirmed': r_conf['n_tests'].mean() if len(r_conf) > 0 else np.nan,
        'pct_multi_pretest': (r_conf['n_pre_tx_tests'] > 1).mean() * 100 if len(r_conf) > 0 else np.nan,
        'pct_interrupted_tx': (r_treat['n_tx_starts'] > 1).mean() * 100 if len(r_treat) > 0 else np.nan,
        'median_system_delay': r_valid['system_delay'].median() if len(r_valid) > 0 else np.nan,
    })
rep_df = pd.DataFrame(rep_metrics)

# ==================================================================
# FIGURE 4: Care-cascade calibration (2-panel)
#   Panel A: Tests per confirmed TB case (histogram + normal fit)
#   Panel B: Average tests per person by loop group (1 = ideal)
# ==================================================================

# -- Panel B data: per-replicate mean tests/person in each loop group --
group_order_care = ["NoLooponly", "TestLooponly", "TestTxfail"]
GROUP_LABELS = {"NoLooponly": "NoLoop only", "TestLooponly": "Test loop only",
                "TestTxfail": "Test + Tx fail"}
if "loop_df" in dir() and "n_tests" in loop_df.columns:
    _care = loop_df[(loop_df["loop_group"].isin(group_order_care))
                    & (loop_df["n_tests"] > 0)]
    _grp = _care.groupby(["replicate", "loop_group"]).agg(
        mean_tests=("n_tests", "mean"),
    )
    avg_tests_per_person = (_grp.groupby("loop_group")
                                .mean()
                                .reindex(group_order_care)["mean_tests"])
else:
    avg_tests_per_person = pd.Series({g: np.nan for g in group_order_care})

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ---------- Panel A: Tests per confirmed TB case ----------
ax = axes[0]
if len(confirmed_df) > 0:
    test_data = confirmed_df["n_tests"].values
    max_tests = min(int(np.percentile(test_data, 99)), 15)
    bins = np.arange(0.5, max_tests + 1.5, 1)

    ax.hist(test_data, bins=bins, color="#3498db",
            edgecolor="white", alpha=0.7, density=True,
            label="Observed")

    mu, sigma = tests_per_confirmed, tests_per_confirmed_std
    x_smooth = np.linspace(0.5, max_tests + 0.5, 200)
    normal_pdf = stats.norm.pdf(x_smooth, mu, sigma)
    ax.plot(x_smooth, normal_pdf, "r-", lw=2.5,
            label=f"Normal fit\n($\\mu$={mu:.1f}, $\\sigma$={sigma:.1f})")

    ax.axvline(mu, color="red", ls="--", lw=1.5, alpha=0.7)
    ax.axvline(tests_per_confirmed_med, color="darkred", ls=":", lw=1.5, alpha=0.7)

    ax.text(0.97, 0.95,
            f"Mean = {mu:.1f}\nMedian = {tests_per_confirmed_med:.0f}\nSD = {sigma:.1f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    ax.set_xlim(0, max_tests + 1)

ax.set_xlabel("Number of tests", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("A. Tests per confirmed TB case",
             fontweight="bold", fontsize=12, loc="left")
ax.legend(fontsize=9, loc="upper left")
ax.grid(axis="y", alpha=0.3)

# ---------- Panel B: Average tests per person by loop group ----------
ax = axes[1]
x = np.arange(len(group_order_care))
shades = ["#e6eef9", "#9bbce0", "#3a78b8"]  # light -> dark blue
bars = ax.bar(x, avg_tests_per_person.values, color=shades,
              edgecolor="#3a78b8", lw=0.8, width=0.6)
for bar, v in zip(bars, avg_tests_per_person.values):
    if pd.notna(v):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.05,
                f"{v:.1f}", ha="center", va="bottom",
                fontsize=11, fontweight="bold")

ax.axhline(1.0, color="red", ls="--", lw=1.2, label="Ideal (1 test/person)")
ax.set_xticks(x)
ax.set_xticklabels([GROUP_LABELS[g] for g in group_order_care],
                   fontsize=10, rotation=15)
ax.set_ylabel("Tests per person", fontsize=11)
ymax = float(np.nanmax(avg_tests_per_person.values)) if len(avg_tests_per_person) else 1.0
ax.set_ylim(0, max(ymax * 1.25, 1.4))
ax.set_title("B. Average tests per person by loop group",
             fontweight="bold", fontsize=12, loc="left")
ax.legend(fontsize=9, loc="upper left")
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.suptitle("Figure 4: Patient pathway cascade calibration\n"
             f"(n = {len(cascade_df):,} active TB episodes, "
             f"{cascade_df.replicate.nunique()} replicates)",
             fontsize=13, fontweight="bold", y=1.04)
plt.tight_layout()
plt.savefig("Fig4_cascade_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved: Fig4_cascade_metrics.png")
print("\n=== Average tests per person ===")
for g in group_order_care:
    v = avg_tests_per_person.get(g, np.nan)
    print(f"  {g:15s} {v:.2f}")
# ── Manuscript-ready statements ──
print('\n' + '=' * 60)
print('MANUSCRIPT-READY STATEMENTS')
print('=' * 60)
print(f'Among simulated active TB episodes:')
print(f'  - Confirmed TB cases required a mean of {tests_per_confirmed:.1f} tests '
      f'(SD {tests_per_confirmed_std:.1f}, median {tests_per_confirmed_med:.0f}).')
print(f'  - {pct_multi_test:.1f}% underwent >1 pre-treatment test.')
print(f'  - Among treated patients, {pct_interrupted_tx:.1f}% had interrupted or')
print(f'    repeated treatment initiation.')
print(f'  - The median system delay was {median_system_delay:.0f} days')
print(f'    (IQR: {p25_system_delay:.0f}–{p75_system_delay:.0f}).')


### Group 4: Intervention Impact -- Reducing Loops vs Expanding Coverage

Should India invest in expanding coverage (ACF), improving diagnostic
accuracy (Dx), or ensuring treatment completion (Tx)? The loop framework
reveals that **reducing loops can substitute for coverage expansion**.


#### Figure 5 -- Intervention Scenarios

Eight scenarios (Baseline, ACF, Dx, Tx, and combinations) starting 2025.
Shows how each intervention shifts loop group distribution and reduces
the fraction of patients trapped in repeated failures.


In [ ]:
# ======================================================================
# FIGURES 5 & 6: INTERVENTION SCENARIOS: How ACF, Dx, Tx shift loop group distribution
#
# Background: DOTS with existing GeneXpert (baseline cascade parameters)
# Interventions: additional improvements from 2025 onward
#
# Scenarios (Table 2):
#   0. Baseline — status quo (DOTS + GeneXpert)
#   1. ACF — active case finding (community screening, contact tracing)
#   2. Dx — diagnostic improvement (Low QoC sensitivity → High QoC level)
#   3. Tx — treatment adherence (digital DOT, patient navigators)
#   4. ACF+Dx
#   5. ACF+Tx
#   6. Dx+Tx
#   7. ACF+Dx+Tx (comprehensive)
#
# EMOD parameter mapping:
#   ACF: acf_coverage = 0.5 (fraction of subclinical TB found)
#   Dx:  Low QoC sensitivity raised from 0.60/0.55 to 0.75/0.65
#        → reduces TBTestNegative and TBTestDefault events
#   Tx:  treatment_fraction raised from 0.55 to 0.85
#        → reduces TBTestDefault (pre-Tx LTFU) and treatment failure
# ======================================================================

loop_df_check = locals().get('loop_df', None)
episode_check = locals().get('episode_df', None)
delay_check = locals().get('delay_df', None)

if loop_df_check is None or episode_check is None or delay_check is None:
    print('⚠ Run cells 32–33 and Figure 2–5 cells first (loop_df, episode_df, delay_df).')
else:
    # ── Baseline metrics from data ──
    n_total = len(loop_df)
    baseline_groups = loop_df['loop_group'].value_counts()
    baseline_pcts = baseline_groups / n_total * 100

    print('=' * 70)
    print('BASELINE LOOP GROUP DISTRIBUTION (Status quo: DOTS + GeneXpert)')
    print('=' * 70)
    for g in ['Latent', 'NoLooponly', 'TestLooponly', 'TestTxfail']:
        pct = baseline_pcts.get(g, 0)
        print(f'  {g:15s}: {baseline_groups.get(g, 0):>6,} ({pct:.1f}%)')

    # Baseline cascade metrics (among active TB = non-Latent)
    active = loop_df[loop_df['loop_group'] != 'Latent']
    n_active = len(active)
    bl_tests_per_case = active['n_tests'].mean()
    bl_pct_multitest = (active['n_tests'] > 1).mean() * 100
    bl_pct_txfail = (active['Txfail'] > 0).mean() * 100
    bl_pct_anyloop = (active['loop_group'].isin(
        ['TestLooponly', 'TestTxfail'])).mean() * 100

    # Baseline transmission shares
    loop_cols = ['screening_loop_days', 'diagnostic_loop_days', 'treatment_loop_days']
    bl_total_loop_days = episode_df[loop_cols].sum().sum()
    bl_total_all_days = episode_df[['baseline_days', 'on_treatment_days'] + loop_cols].sum().sum()
    bl_loop_share = bl_total_loop_days / bl_total_all_days * 100
    bl_screen_days = episode_df['screening_loop_days'].sum()
    bl_dx_days = episode_df['diagnostic_loop_days'].sum()
    bl_tx_days = episode_df['treatment_loop_days'].sum()

    # ── Define intervention scenarios ──
    # Intervention effects mapped to EMOD parameters:
    #   ACF:  acf_coverage fraction → catches subclinical cases before looping
    #         Mechanism: converts TestLoop/TestTxfail → NoLoop
    #   Dx:   Low QoC sensitivity → High QoC level
    #         Effect: reduces P(TBTestNegative) and P(TBTestDefault)
    #         Effectiveness = 1 - (Low_sensitivity / High_sensitivity)
    #         SmPos: 1 - 0.60/0.75 = 0.20; SmNeg: 1 - 0.55/0.65 = 0.15
    #         Average effect on test loops: ~0.5 reduction
    #   Tx:   treatment_fraction 0.65 → 0.85
    #         Effect: reduces P(TBTestDefault from pre-Tx LTFU) by (0.85-0.65)/0.35 ≈ 0.57
    #         Also reduces treatment failure/relapse

    dx_effect = 0.50   # avg reduction in diagnostic loop probability
    tx_effect = 0.67   # reduction in pre-Tx LTFU (tf: 0.55→0.85)
    acf_effect = 0.50  # fraction of looping cases caught early by ACF

    scenarios = {
        'Baseline': {
            'acf': 0.0, 'dx': 0.0, 'tx': 0.0,
            'acf_coverage': 0.0,
            'sens_low': (0.65, 0.60),
            'tf_low': 0.55,
            'color': '#2c3e50', 'marker': 'o',
            'label': 'Status quo',
        },
        'ACF': {
            'acf': acf_effect, 'dx': 0.0, 'tx': 0.0,
            'acf_coverage': 0.50,
            'sens_low': (0.65, 0.60),
            'tf_low': 0.55,
            'color': '#2ca02c', 'marker': 's',
            'label': 'Active case finding',
        },
        'Dx': {
            'acf': 0.0, 'dx': dx_effect, 'tx': 0.0,
            'acf_coverage': 0.0,
            'sens_low': (0.75, 0.65),  # raised to High QoC
            'tf_low': 0.55,
            'color': '#1f77b4', 'marker': '^',
            'label': 'Dx: Low QoC → High QoC sensitivity',
        },
        'Tx': {
            'acf': 0.0, 'dx': 0.0, 'tx': tx_effect,
            'acf_coverage': 0.0,
            'sens_low': (0.65, 0.60),
            'tf_low': 0.85,
            'color': '#d62728', 'marker': 'D',
            'label': 'Tx adherence improvement',
        },
        'ACF+Dx': {
            'acf': acf_effect, 'dx': dx_effect, 'tx': 0.0,
            'acf_coverage': 0.50,
            'sens_low': (0.75, 0.65),
            'tf_low': 0.55,
            'color': '#17becf', 'marker': 'v',
            'label': 'Case finding + diagnostics',
        },
        'ACF+Tx': {
            'acf': acf_effect, 'dx': 0.0, 'tx': tx_effect,
            'acf_coverage': 0.50,
            'sens_low': (0.65, 0.60),
            'tf_low': 0.85,
            'color': '#bcbd22', 'marker': '<',
            'label': 'Case finding + treatment',
        },
        'Dx+Tx': {
            'acf': 0.0, 'dx': dx_effect, 'tx': tx_effect,
            'acf_coverage': 0.0,
            'sens_low': (0.75, 0.65),
            'tf_low': 0.85,
            'color': '#9467bd', 'marker': '>',
            'label': 'Diagnostics + treatment',
        },
        'ACF+Dx+Tx': {
            'acf': acf_effect, 'dx': dx_effect, 'tx': tx_effect,
            'acf_coverage': 0.50,
            'sens_low': (0.75, 0.65),
            'tf_low': 0.85,
            'color': '#e377c2', 'marker': 'P',
            'label': 'Comprehensive',
        },
    }

    # ── Project loop group distribution under each scenario ──
    scenario_results = []

    for sc_name, sc in scenarios.items():
        r_acf = sc['acf']
        r_dx = sc['dx']
        r_tx = sc['tx']

        # For each individual, recalculate effective loop triggers
        df = active.copy()

        # Effective trigger counts after intervention
        # Dx reduces false-neg retests (TBTestNegative, TBTestDefault)
        df['eff_testneg'] = df['Testneg'] * (1 - r_dx)
        df['eff_testdefault'] = df['Testdefault'] * (1 - r_dx)
        # Tx reduces treatment failure
        df['eff_txfail'] = df['Txfail'] * (1 - r_tx)

        # ACF: cases found by ACF skip the care-seeking loop
        # Converts some TestLoop/TestTxfail cases to NoLoop
        np.random.seed(42)  # reproducibility
        acf_convert_mask = (df['loop_group'].isin(['TestLooponly', 'TestTxfail'])) & \
                          (np.random.random(len(df)) < r_acf * 0.3)

        # Reclassify loop groups based on effective counts
        def reclassify(row):
            tn = row['eff_testneg']
            td = row['eff_testdefault']
            tf = row['eff_txfail']
            has_test = (tn >= 1) or (td >= 1)
            has_tx = (tf >= 1)
            if has_test and has_tx:
                return 'TestTxfail'
            if has_test and not has_tx:
                return 'TestLooponly'
            if not has_test and has_tx:
                return 'Txfailonly'
            return 'NoLooponly'

        df['new_group'] = df.apply(reclassify, axis=1)
        df.loc[acf_convert_mask, 'new_group'] = 'NoLooponly'

        # Count new distribution
        new_groups = df['new_group'].value_counts()

        # Project cascade metrics
        proj_tests = bl_tests_per_case * (1 - r_dx * 0.4)
        if r_acf > 0:
            proj_tests += r_acf * 0.2

        # Transmission: loop days reduced
        proj_screen_days = bl_screen_days * (1 - r_dx * 0.5 - r_acf * 0.3)
        proj_dx_days = bl_dx_days * (1 - r_dx)
        proj_tx_days = bl_tx_days * (1 - r_tx)
        proj_loop_days = max(0, proj_screen_days) + max(0, proj_dx_days) + max(0, proj_tx_days)
        proj_loop_share = proj_loop_days / bl_total_all_days * 100

        # Incidence reduction
        # --- Transmission-feedback-aware incidence reduction ---
        # Direct: fraction of total infectious time removed by intervention
        direct_red = max(0.0, (bl_total_loop_days - proj_loop_days) / bl_total_all_days)
        # Add ACF effect on subclinical infectious time (ACF shortens subclinical period
        # by ~50%; subclinical is ~30% of total infectious time at baseline)
        SUBCLIN_SHARE = 0.30
        ACF_SUBCLIN_EFFECT = 0.50
        direct_red += SUBCLIN_SHARE * ACF_SUBCLIN_EFFECT * r_acf
        # R0 amplification: at endemic equilibrium, reducing per-case infectious time
        # by fraction d propagates through the transmission chain. Empirical TB
        # transmission models (Dye et al., Houben et al.) show ~3x amplification.
        R_AMP = 1.5
        inc_reduction = (1 - (1 - min(direct_red, 0.85))**R_AMP) * 100

        result = {
            'scenario': sc_name,
            'acf': r_acf, 'dx': r_dx, 'tx': r_tx,
            'acf_coverage': sc['acf_coverage'],
            'sens_low_sp': sc['sens_low'][0],
            'sens_low_sn': sc['sens_low'][1],
            'tf_low': sc['tf_low'],
            'NoLooponly': new_groups.get('NoLooponly', 0) / n_active * 100,
            'TestLooponly': new_groups.get('TestLooponly', 0) / n_active * 100,
            'Txfailonly': new_groups.get('Txfailonly', 0) / n_active * 100,
            'TestTxfail': new_groups.get('TestTxfail', 0) / n_active * 100,
            'pct_anyloop': (new_groups.get('TestLooponly', 0) +
                           new_groups.get('Txfailonly', 0) +
                           new_groups.get('TestTxfail', 0)) / n_active * 100,
            'tests_per_case': proj_tests,
            'loop_transmission_share': proj_loop_share,
            'inc_reduction_pct': inc_reduction,
            'inc_reduction': inc_reduction,  # alias used by Figs D/E/F/P1-P6
            'color': sc['color'],
            'marker': sc['marker'],
            'label': sc['label'],
        }
        scenario_results.append(result)

    sc_df = pd.DataFrame(scenario_results)

    # ======================================================================
    # FIGURE 6: Intervention scenarios — loop group distribution shifts
    # ======================================================================


#### Figure 5a/5b -- Intervention Impact Waterfall Charts

How each intervention (ACF, Dx, Tx, and combinations) changes
cascade events and loop group distribution relative to baseline.


In [ ]:
# ======================================================================
# FIGURE 5a: Intervention Impact on Cascade Events
# FIGURE 5b: Intervention Impact on Loop Groups
#
# Waterfall charts showing % change from baseline for each scenario
# ======================================================================

if loop_df is not None and len(loop_df) > 0:

    # ── Baseline counts ──
    event_cols = ['NewInf', 'PreSym', 'Act', 'Testpos', 'Testneg', 'Testdefault', 'Txfail', 'Clear', 'Death']
    event_labels = ['NewInf', 'PreSym', 'Act', 'TestposTx', 'Testneg', 'Testdefault', 'Txfail', 'Clear', 'Death']
    baseline_events = loop_df[event_cols].sum()

    group_order = ['Latent', 'NoLooponly', 'TestLooponly', 'TestTxfail']
    group_labels_short = ['Latent', 'NoLoop', 'TestingLoop', 'TestTxLoop']
    baseline_groups = loop_df['loop_group'].value_counts().reindex(group_order, fill_value=0)

    active = loop_df[loop_df['loop_group'] != 'Latent'].copy()

    # ── Scenario definitions (same as Figure 5) ──
    scenarios_list = [
        ('ACF',      {'acf': 0.30, 'dx': 0.0, 'tx': 0.0}),
        ('Dx',       {'acf': 0.0,  'dx': 0.50, 'tx': 0.0}),
        ('Tx',       {'acf': 0.0,  'dx': 0.0,  'tx': 0.50}),
        ('ACFDx',    {'acf': 0.30, 'dx': 0.50, 'tx': 0.0}),
        ('ACFDxTx',  {'acf': 0.30, 'dx': 0.50, 'tx': 0.50}),
    ]

    def simulate_scenario(df_active, r_acf, r_dx, r_tx, baseline_events, baseline_groups):
        """Simulate intervention effects on events and loop groups."""
        df = df_active.copy()

        # --- Event changes ---
        # ACF finds more cases -> more NewInf detected, more Act, more tests
        # Dx reduces false negatives and defaults
        # Tx reduces treatment failures
        new_events = baseline_events.copy().astype(float)

        # NewInf reduction: more found early = fewer ongoing transmissions
        transmission_reduction = r_acf * 0.07 + r_dx * 0.05 + r_tx * 0.08
        new_events['NewInf'] *= (1 - transmission_reduction)
        new_events['PreSym'] *= (1 - transmission_reduction)
        new_events['Act'] *= (1 - transmission_reduction)

        # Diagnostic events
        new_events['Testneg'] *= (1 - r_dx * 0.9)     # Dx reduces false negatives
        new_events['Testdefault'] *= (1 - r_dx * 0.6)  # Dx reduces test defaults
        new_events['Testpos'] *= (1 - transmission_reduction * 0.5)  # slightly fewer due to fewer cases

        # Treatment events
        new_events['Txfail'] *= (1 - r_tx * 0.9)  # Tx reduces treatment failure

        # Resolution
        new_events['Clear'] *= (1 - transmission_reduction * 0.3)
        new_events['Death'] *= (1 - transmission_reduction - r_tx * 0.05)

        # --- Loop group changes ---
        df['eff_testneg'] = df['Testneg'] * (1 - r_dx)
        df['eff_testdefault'] = df['Testdefault'] * (1 - r_dx)
        df['eff_txfail'] = df['Txfail'] * (1 - r_tx)

        np.random.seed(42)
        acf_mask = (df['loop_group'].isin(['TestLooponly', 'TestTxfail'])) & \
                   (np.random.random(len(df)) < r_acf * 0.3)

        def reclassify(row):
            tn, td, tf = row['eff_testneg'], row['eff_testdefault'], row['eff_txfail']
            if (tn > 1 and tf > 1) or (td > 1 and tf > 1):
                return 'TestTxfail'
            elif tn == 0 and td == 0 and tf > 1:
                return 'Txfailonly'
            elif (tn > 1 or td > 1) and tf <= 1:
                return 'TestLooponly'
            else:
                return 'NoLooponly'

        df['new_group'] = df.apply(reclassify, axis=1)
        df.loc[acf_mask, 'new_group'] = 'NoLooponly'

        new_grp_counts = df['new_group'].value_counts()
        # Scale by transmission reduction for Latent
        latent_baseline = baseline_groups.get('Latent', 0)
        new_groups_full = pd.Series({
            'Latent': latent_baseline * (1 - transmission_reduction),
            'NoLooponly': new_grp_counts.get('NoLooponly', 0),
            'TestLooponly': new_grp_counts.get('TestLooponly', 0),
            'TestTxfail': new_grp_counts.get('TestTxfail', 0),
        })

        return new_events, new_groups_full

    # ── Compute changes for all scenarios ──
    event_changes = {}
    group_changes = {}

    for sc_name, sc_params in scenarios_list:
        new_ev, new_grp = simulate_scenario(
            active, sc_params['acf'], sc_params['dx'], sc_params['tx'],
            baseline_events, baseline_groups
        )
        # % change from baseline
        ev_pct = ((new_ev - baseline_events) / baseline_events * 100).fillna(0)
        ev_abs = (new_ev - baseline_events).fillna(0)
        event_changes[sc_name] = {'pct': ev_pct, 'abs': ev_abs}

        grp_bl = baseline_groups.reindex(group_order, fill_value=0).astype(float)
        grp_new = new_grp.reindex(group_order, fill_value=0).astype(float)
        grp_pct = ((grp_new - grp_bl) / grp_bl * 100).fillna(0)
        grp_abs = (grp_new - grp_bl).fillna(0)
        group_changes[sc_name] = {'pct': grp_pct, 'abs': grp_abs}

    # ======================================================================
    # FIGURE 5a: Impact on Cascade Events
    # ======================================================================
    n_sc = len(scenarios_list)
    ncols = 3
    nrows = (n_sc + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
    axes_flat = axes.flatten()

    event_colors = ['#FFD700', '#CDDC39', '#8BC34A', '#26A69A', '#00897B',
                    '#5C6BC0', '#3F51B5', '#9E9E9E', '#424242']

    for idx, (sc_name, _) in enumerate(scenarios_list):
        ax = axes_flat[idx]
        abs_vals = event_changes[sc_name]['abs']
        pct_vals = event_changes[sc_name]['pct']

        bars = ax.bar(event_labels, abs_vals.values, color=event_colors, edgecolor='white', width=0.7)

        for bar, pct in zip(bars, pct_vals.values):
            h = bar.get_height()
            if not np.isnan(h) and abs(h) > 1:
                y_pos = h - 30 if h < 0 else h + 10
                ax.text(bar.get_x() + bar.get_width()/2, y_pos,
                        f'{pct:.0f}%', ha='center', va='top' if h < 0 else 'bottom',
                        fontsize=8, fontweight='bold',
                        color='darkred' if pct < 0 else 'darkgreen')

        ax.set_title(sc_name, fontsize=13, fontweight='bold')
        ax.set_ylabel('Number of events' if idx % ncols == 0 else '')
        ax.axhline(0, color='black', lw=0.5)
        ax.grid(axis='y', alpha=0.2)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)

    # Hide unused axes
    for idx in range(len(scenarios_list), len(axes_flat)):
        axes_flat[idx].axis('off')

    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=l) for c, l in zip(event_colors, event_labels)]
    axes_flat[-1].legend(handles=legend_elements, loc='center', fontsize=10,
                         title='Group', title_fontsize=11, frameon=True)
    axes_flat[-1].axis('off')

    fig.suptitle('Figure 5a: Intervention Impact on Cascade Events\n(Change from Baseline)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()

    # ======================================================================
    # FIGURE 5b: Impact on Loop Groups
    # ======================================================================
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
    axes_flat = axes.flatten()

    grp_colors = ['#FFD700', '#66BB6A', '#00897B', '#283593', '#4A148C']

    for idx, (sc_name, _) in enumerate(scenarios_list):
        ax = axes_flat[idx]
        abs_vals = group_changes[sc_name]['abs'].reindex(group_order, fill_value=0)
        pct_vals = group_changes[sc_name]['pct'].reindex(group_order, fill_value=0)

        bars = ax.bar(group_labels_short, abs_vals.values,
                      color=grp_colors[:len(group_order)], edgecolor='white', width=0.6)

        for bar, pct in zip(bars, pct_vals.values):
            h = bar.get_height()
            if not np.isnan(h) and abs(h) > 1:
                y_pos = h - 20 if h < 0 else h + 10
                ax.text(bar.get_x() + bar.get_width()/2, y_pos,
                        f'{pct:.0f}%', ha='center', va='top' if h < 0 else 'bottom',
                        fontsize=9, fontweight='bold',
                        color='darkred' if pct < 0 else 'darkgreen')

        ax.set_title(sc_name, fontsize=13, fontweight='bold')
        ax.set_ylabel('Number of people' if idx % ncols == 0 else '')
        ax.axhline(0, color='black', lw=0.5)
        ax.grid(axis='y', alpha=0.2)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right', fontsize=9)

    for idx in range(len(scenarios_list), len(axes_flat)):
        axes_flat[idx].axis('off')

    legend_elements = [Patch(facecolor=c, label=l)
                       for c, l in zip(grp_colors[:len(group_order)], group_labels_short)]
    axes_flat[-1].legend(handles=legend_elements, loc='center', fontsize=10,
                         title='Group', title_fontsize=11, frameon=True)
    axes_flat[-1].axis('off')

    fig.suptitle('Figure 5b: Intervention Impact on Loop Groups\n(Change from Baseline)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('Fig5b_loopgroup_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: Fig5b_loopgroup_waterfall.png')

    # ── Summary table ──
    print('\n=== Intervention Impact Summary (% change from baseline) ===')
    print(f'{"Scenario":<12s} | {"NewInf":>7s} {"Testneg":>8s} {"Testdef":>8s} {"Txfail":>7s} {"Death":>6s} | {"Latent":>7s} {"NoLoop":>7s} {"TestLp":>7s} {"TstTxF":>7s}')
    print('-' * 95)
    for sc_name, _ in scenarios_list:
        ev = event_changes[sc_name]['pct']
        gr = group_changes[sc_name]['pct']
        print(f'{sc_name:<12s} | {ev["NewInf"]:>6.0f}% {ev["Testneg"]:>7.0f}% {ev["Testdefault"]:>7.0f}% {ev["Txfail"]:>6.0f}% {ev["Death"]:>5.0f}% | '
              f'{gr.get("Latent",0):>6.0f}% {gr.get("NoLooponly",0):>6.0f}% {gr.get("TestLooponly",0):>6.0f}% {gr.get("TestTxfail",0):>6.0f}%')
else:
    print('Run data loading cells first.')


#### Figure 5c -- Intervention Impact on Delay Phases

How each intervention changes subclinical, patient delay,
and health system delay person-years by loop group.
This directly connects interventions to transmission reduction.


In [ ]:
# ======================================================================
# FIGURE 5c: Intervention Impact on Delay Phases
#
# Waterfall charts showing how each intervention changes the three
# delay phases (Subclinical, Patient Delay, Health System Delay)
# by loop group -- the key link between loops and transmission.
# ======================================================================

if delay_df is not None and len(delay_df) > 0:
    from matplotlib.patches import Patch

    delay_cols_fig = ['subclinical_days', 'patient_delay_days', 'health_system_delay_days']
    delay_labels_fig = ['Subclinical', 'Patient delay', 'Health system delay']
    delay_colors_fig = ['#f5b7b1', '#e67e22', '#c0392b']

    grp_order_fig = ['NoLooponly', 'TestLooponly', 'TestTxfail']
    grp_labels_fig = ['NoLoop', 'TestingLoop', 'TestTxLoop']

    # Baseline delay totals (person-years) by loop group
    baseline_delay = {}
    for grp in grp_order_fig:
        gdf = delay_df[delay_df['loop_group'] == grp]
        baseline_delay[grp] = {col: gdf[col].sum() / 365.25 for col in delay_cols_fig}
    baseline_delay['Total'] = {col: delay_df[col].sum() / 365.25 for col in delay_cols_fig}

    scenarios_delay = [
        ('ACF',      {'acf': 0.30, 'dx': 0.0, 'tx': 0.0}),
        ('Dx',       {'acf': 0.0,  'dx': 0.50, 'tx': 0.0}),
        ('Tx',       {'acf': 0.0,  'dx': 0.0,  'tx': 0.50}),
        ('ACFDx',    {'acf': 0.30, 'dx': 0.50, 'tx': 0.0}),
        ('ACFDxTx',  {'acf': 0.30, 'dx': 0.50, 'tx': 0.50}),
    ]

    def project_delays(bl_delay, r_acf, r_dx, r_tx):
        results = {}
        for grp in grp_order_fig:
            bl = bl_delay[grp]
            nw = {}
            nw['subclinical_days'] = bl['subclinical_days'] * (1 - r_acf * 0.5)
            nw['patient_delay_days'] = bl['patient_delay_days'] * (1 - r_acf * 0.7)
            if grp == 'NoLooponly':
                nw['health_system_delay_days'] = bl['health_system_delay_days'] * (1 - r_dx * 0.3)
            elif grp == 'TestLooponly':
                nw['health_system_delay_days'] = bl['health_system_delay_days'] * (1 - r_dx * 0.7)
            elif grp == 'TestTxfail':
                nw['health_system_delay_days'] = bl['health_system_delay_days'] * (1 - r_dx * 0.4 - r_tx * 0.4)
            results[grp] = nw
        results['Total'] = {col: sum(results[g][col] for g in grp_order_fig) for col in delay_cols_fig}
        return results

    # ── Figure 5c ──
    ncols = 3
    nrows = 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 10))
    axes_flat = axes.flatten()

    x_labels = grp_labels_fig + ['Total']
    x = np.arange(len(x_labels))

    for idx, (sc_name, sc_params) in enumerate(scenarios_delay):
        ax = axes_flat[idx]
        proj = project_delays(baseline_delay, sc_params['acf'], sc_params['dx'], sc_params['tx'])

        width = 0.6
        for gi, (grp, xlabel) in enumerate(zip(grp_order_fig + ['Total'], x_labels)):
            bl = baseline_delay[grp]
            nw = proj[grp]
            bottom = 0
            for col, label, color in zip(delay_cols_fig, delay_labels_fig, delay_colors_fig):
                change = nw[col] - bl[col]
                bar_kw = {'label': label} if gi == 0 else {}
                ax.bar(x[gi], change, width, bottom=bottom, color=color,
                       edgecolor='white', **bar_kw)
                bottom += change

            total_bl = sum(bl[c] for c in delay_cols_fig)
            total_nw = sum(nw[c] for c in delay_cols_fig)
            if total_bl > 0:
                pct = (total_nw - total_bl) / total_bl * 100
                ax.text(x[gi], bottom - 5, f'{pct:.0f}%',
                        ha='center', va='top', fontsize=9, fontweight='bold',
                        color='darkred' if pct < 0 else 'darkgreen')

        ax.set_title(sc_name, fontsize=13, fontweight='bold')
        ax.set_ylabel('Change in person-years' if idx % ncols == 0 else '')
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=25, ha='right', fontsize=9)
        ax.axhline(0, color='black', lw=0.5)
        ax.grid(axis='y', alpha=0.2)
        if idx == 0:
            ax.legend(fontsize=8, loc='lower left')

    for idx in range(len(scenarios_delay), len(axes_flat)):
        axes_flat[idx].axis('off')

    # Summary in empty panel
    ax_summary = axes_flat[-1]
    summary_lines = ['Summary (% total delay reduction):\n']
    for sc_name, sc_params in scenarios_delay:
        proj = project_delays(baseline_delay, sc_params['acf'], sc_params['dx'], sc_params['tx'])
        bl_total = sum(baseline_delay['Total'][c] for c in delay_cols_fig)
        nw_total = sum(proj['Total'][c] for c in delay_cols_fig)
        pct = (nw_total - bl_total) / bl_total * 100
        summary_lines.append(f'{sc_name:<10s}: {pct:+.1f}%')
    ax_summary.text(0.1, 0.5, '\n'.join(summary_lines), transform=ax_summary.transAxes,
                    fontsize=11, fontfamily='monospace', va='center',
                    bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.9))

    fig.suptitle('Figure 5c: Intervention Impact on Infectious Time by Delay Phase\n'
                 '(Change in person-years from Baseline, by loop group)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('Fig5c_delay_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: Fig5c_delay_waterfall.png')

    # Print breakdown
    print('\n=== Delay Phase Changes by Loop Group ===')
    for sc_name, sc_params in scenarios_delay:
        proj = project_delays(baseline_delay, sc_params['acf'], sc_params['dx'], sc_params['tx'])
        print(f'\n{sc_name}:')
        for grp, glabel in zip(grp_order_fig + ['Total'], grp_labels_fig + ['TOTAL']):
            bl = baseline_delay[grp]
            nw = proj[grp]
            bl_t = sum(bl[c] for c in delay_cols_fig)
            nw_t = sum(nw[c] for c in delay_cols_fig)
            pct = (nw_t - bl_t) / bl_t * 100 if bl_t > 0 else 0
            print(f'  {glabel:<12s}  subclin: {bl["subclinical_days"]:.0f} -> {nw["subclinical_days"]:.0f} py  '
                  f'patient: {bl["patient_delay_days"]:.0f} -> {nw["patient_delay_days"]:.0f} py  '
                  f'HS: {bl["health_system_delay_days"]:.0f} -> {nw["health_system_delay_days"]:.0f} py  '
                  f'({pct:+.1f}%)')
else:
    print('Run delay decomposition cell (Figure 2) first.')


### Figure DEF -- Triple-Lens Heatmap (combined D + E + F)

Single figure consolidating Figures D, E, F as 3 stacked panels with a shared
x-axis (8 scenarios) and shared colormap. Each panel uses a different lens for
% reduction vs Baseline:

- **Top (loop groups):** who is failing the cascade
- **Middle (delay phases):** where in the timeline transmission is lost
- **Bottom (resource waste):** how much system waste is removed

Reading down a column shows the full mechanistic profile of that scenario
across all three lenses simultaneously.


In [ ]:
# ======================================================================
# FIGURE DEF: Triple-Lens Heatmap (combined Figs D + E + F)
# 3 panels sharing x-axis (scenarios), shared colorbar
# ======================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec

if "scenario_results" not in dir() or "loop_df" not in dir() or "episode_df" not in dir():
    raise RuntimeError("Need scenario_results, loop_df, episode_df. Run earlier cells first.")

scen_df = (pd.DataFrame(scenario_results)
           if isinstance(scenario_results, list) else scenario_results.copy())
scen_order = ["Baseline", "ACF", "Dx", "Tx",
              "ACF+Dx", "ACF+Tx", "Dx+Tx", "ACF+Dx+Tx"]
if "scenario" in scen_df.columns:
    scen_df = scen_df.set_index("scenario")
scen_df = scen_df.reindex(scen_order)

# ===================================================================
# PANEL 1 (Loop Groups) -- copied from Figure D logic
# ===================================================================
n_total = len(loop_df)
n_latent = (loop_df["loop_group"] == "Latent").sum()
n_active_bl = n_total - n_latent

inc_red_col = ("inc_reduction" if "inc_reduction" in scen_df.columns
               else ("inc_reduction_pct" if "inc_reduction_pct" in scen_df.columns
                     else None))

abs_loop = []
for _, r in scen_df.reset_index().iterrows():
    sc = r["scenario"] if "scenario" in r else r.name
    inc_red = r[inc_red_col] if inc_red_col else 0.0
    active_total = n_active_bl * max(0.0, 1 - inc_red / 100.0)
    acf = r.get("acf", 0.0)
    latent_count = n_latent * (1 - 0.30 * acf)
    abs_loop.append({
        "scenario":     sc,
        "NoLooponly":   active_total * float(r.get("NoLooponly",   0)) / 100,
        "TestLooponly": active_total * float(r.get("TestLooponly", 0)) / 100,
        "TestTxfail":   active_total * float(r.get("TestTxfail",   0)) / 100,
        "Latent":       latent_count,
    })
abs_loop_df = pd.DataFrame(abs_loop).set_index("scenario").reindex(scen_order)
loop_groups = ["NoLooponly", "TestLooponly", "TestTxfail", "Latent"]
M_loop = ((abs_loop_df.loc["Baseline"] - abs_loop_df) /
          abs_loop_df.loc["Baseline"] * 100)[loop_groups].T

# ===================================================================
# PANEL 2 (Delay Phases) -- copied from Figure E logic
# ===================================================================
def _col_or_zero(df, col):
    return df[col].fillna(0) if col in df.columns else pd.Series(0.0, index=df.index)

if "delay_df" in dir():
    if "person_id" in delay_df.columns:
        pdelay = delay_df.groupby("person_id").mean(numeric_only=True)
    else:
        pdelay = delay_df.copy()
    bl_subclin = _col_or_zero(pdelay, "subclinical_days").sum()
    bl_patient = _col_or_zero(pdelay, "patient_delay_days").sum()
    if bl_subclin < 1:
        bl_subclin = len(pdelay) * 90.0
    if bl_patient < 1:
        bl_patient = len(pdelay) * 60.0
else:
    bl_subclin = n_active_bl * 90.0
    bl_patient = n_active_bl * 60.0

bl_hs = (episode_df["diagnostic_loop_days"].sum()
         + episode_df["treatment_loop_days"].sum())

ACF_EFF_SUBCLIN = 0.50
ACF_EFF_PATIENT = 0.70
DX_EFF_HS = 0.50
TX_EFF_HS = 0.40

scen_flags = {
    "Baseline":   {"acf": 0, "dx": 0, "tx": 0},
    "ACF":        {"acf": 1, "dx": 0, "tx": 0},
    "Dx":         {"acf": 0, "dx": 1, "tx": 0},
    "Tx":         {"acf": 0, "dx": 0, "tx": 1},
    "ACF+Dx":     {"acf": 1, "dx": 1, "tx": 0},
    "ACF+Tx":     {"acf": 1, "dx": 0, "tx": 1},
    "Dx+Tx":      {"acf": 0, "dx": 1, "tx": 1},
    "ACF+Dx+Tx":  {"acf": 1, "dx": 1, "tx": 1},
}
share_dx = (episode_df["diagnostic_loop_days"].sum() / max(1, bl_hs))
share_tx = 1 - share_dx

abs_delay = []
for sc in scen_order:
    f = scen_flags[sc]
    sub_v = bl_subclin * (1 - ACF_EFF_SUBCLIN * f["acf"])
    pat_v = bl_patient * (1 - ACF_EFF_PATIENT * f["acf"])
    hs_red = share_dx * DX_EFF_HS * f["dx"] + share_tx * TX_EFF_HS * f["tx"]
    hs_v = bl_hs * (1 - hs_red)
    abs_delay.append({"scenario": sc, "Subclinical": sub_v,
                      "Patient delay": pat_v, "Health-system delay": hs_v})
abs_delay_df = pd.DataFrame(abs_delay).set_index("scenario")
M_delay = ((abs_delay_df.loc["Baseline"] - abs_delay_df) /
           abs_delay_df.loc["Baseline"] * 100).T

# ===================================================================
# PANEL 3 (Resource Waste) -- copied from Figure F logic
# ===================================================================
active = loop_df[loop_df["loop_group"] != "Latent"].copy()
n_testneg = _col_or_zero(active, "Testneg")
n_testdef = _col_or_zero(active, "Testdefault")
bl_wasted_tests = float((n_testneg + n_testdef).sum())
n_txfail = _col_or_zero(active, "Txfail")
DRUG_MO_FAIL = 6.0
bl_wasted_drugmo = float(n_txfail.sum() * DRUG_MO_FAIL)
bl_excess_days = float(episode_df["screening_loop_days"].sum()
                       + episode_df["diagnostic_loop_days"].sum()
                       + episode_df["treatment_loop_days"].sum())

ACF_DX, DX_DX, TX_DX = 0.20, 0.55, 0.0
ACF_TX, DX_TX, TX_TX = 0.10, 0.0, 0.60
ACF_TIME, DX_TIME, TX_TIME = 0.30, 0.40, 0.30

def composite(*fs):
    keep = 1.0
    for f in fs:
        keep *= (1 - f)
    return 1 - keep

abs_res = []
for sc in scen_order:
    f = scen_flags[sc]
    a, d, t = f["acf"], f["dx"], f["tx"]
    dx_red = composite(ACF_DX * a, DX_DX * d, TX_DX * t)
    tx_red = composite(ACF_TX * a, DX_TX * d, TX_TX * t)
    time_red = composite(ACF_TIME * a, DX_TIME * d, TX_TIME * t)
    abs_res.append({
        "scenario": sc,
        "Wasted dx tests":    bl_wasted_tests * (1 - dx_red),
        "Wasted drug-months": bl_wasted_drugmo * (1 - tx_red),
        "Excess person-time": bl_excess_days * (1 - time_red),
    })
abs_res_df = pd.DataFrame(abs_res).set_index("scenario")
M_res = ((abs_res_df.loc["Baseline"] - abs_res_df) /
         abs_res_df.loc["Baseline"].replace(0, np.nan) * 100).T

# ===================================================================
# COMBINED PLOT
# ===================================================================
cmap = LinearSegmentedColormap.from_list(
    "redgreen", ["#b22222", "#f5f5f5", "#1a7f3a"], N=256)
all_vals = np.concatenate([M_loop.values.flatten(),
                           M_delay.values.flatten(),
                           M_res.values.flatten()])
vmax = max(5.0, float(np.nanmax(np.abs(all_vals))))

fig = plt.figure(figsize=(13, 10.5))
gs = GridSpec(3, 2, height_ratios=[4, 3, 3], width_ratios=[40, 1],
              hspace=0.18, wspace=0.04)

panels = [
    ("Loop group",   M_loop,
     ["NoLoop only", "Test loop only", "Test + Tx fail", "Latent reactivation"],
     "(A) Who fails the cascade"),
    ("Delay phase",  M_delay,
     ["Subclinical", "Patient delay", "Health-system delay"],
     "(B) Where in the timeline"),
    ("Wasted resource", M_res,
     ["Wasted dx tests", "Wasted drug-months", "Excess person-time"],
     "(C) System waste removed"),
]

for k, (ylabel, M, ylabs, subtitle) in enumerate(panels):
    ax = fig.add_subplot(gs[k, 0])
    im = ax.imshow(M.values, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_yticks(range(len(ylabs)))
    ax.set_yticklabels(ylabs, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(subtitle, fontsize=11, loc="left", pad=4, fontweight="bold")

    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i, j]
            if np.isnan(v):
                txt, col = "n/a", "gray"
            else:
                txt = ("+" if v >= 0 else "") + str(int(round(v))) + "%"
                col = "white" if abs(v) > vmax * 0.55 else "black"
            ax.text(j, i, txt, ha="center", va="center",
                    fontsize=9.5, color=col, fontweight="bold")

    ax.set_xticks(range(M.shape[1]))
    if k < 2:
        ax.set_xticklabels([])
    else:
        ax.set_xticklabels(M.columns.tolist(), rotation=25, ha="right",
                           fontsize=10)
        ax.set_xlabel("Intervention scenario",
                      fontsize=11, labelpad=8)

    ax.set_xticks(np.arange(-.5, M.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, M.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="minor", length=0)

# Single shared colorbar
cax = fig.add_subplot(gs[:, 1])
cb = fig.colorbar(im, cax=cax)
cb.set_label("% reduction vs Baseline", fontsize=10)

fig.suptitle("Figure DEF: Triple-Lens View of Intervention Impact\n"
             "(% reduction vs Baseline across loop / delay / resource lenses)",
             fontsize=13, fontweight="bold", y=0.995)

os.makedirs("figures", exist_ok=True)
out_path = os.path.join("figures", "FigDEF_combined_triple_lens.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)


### Policy Figure P1 -- Path to End-TB 2035

Two-panel figure compressing the temporal policy story:
- **Top:** projected incidence trajectory 2025-2035 for all 8 scenarios with WHO targets overlaid
- **Bottom:** cumulative cases averted 2025-2035 per scenario

Trajectories use an exponential approach-to-endpoint model anchored to the scenario
endpoint incidence reductions from the calibrated baseline.


In [ ]:
# ======================================================================
# POLICY FIGURE P1: Path to End-TB 2035 (Trajectory + Cumulative Impact)
# ======================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "scenario_results" not in dir():
    raise RuntimeError("Need scenario_results from Figure 5 cell.")

scen_df = (pd.DataFrame(scenario_results)
           if isinstance(scenario_results, list) else scenario_results.copy())
scen_order = ["Baseline", "ACF", "Dx", "Tx",
              "ACF+Dx", "ACF+Tx", "Dx+Tx", "ACF+Dx+Tx"]
scen_df = scen_df.set_index("scenario").reindex(scen_order)

# India baseline incidence ~210/100K (2024); approach-to-endpoint over 10 years
INC_2025 = 210.0
years = np.arange(2025, 2036)
k = 0.30  # rate constant -> ~95% of effect by year 10

inc_red_col = "inc_reduction" if "inc_reduction" in scen_df.columns else None

trajectories = {}
for sc in scen_order:
    inc_red = scen_df.loc[sc, inc_red_col] / 100.0 if inc_red_col else 0.0
    inc_endpoint = INC_2025 * (1 - inc_red)
    traj = inc_endpoint + (INC_2025 - inc_endpoint) * np.exp(-k * (years - 2025))
    trajectories[sc] = traj

# Year reaching 50% and 90% reduction
targets_pct = {"50%": 0.50, "90% (End-TB)": 0.90}
year_reached = {sc: {} for sc in scen_order}
for sc, traj in trajectories.items():
    for tname, t in targets_pct.items():
        target_inc = INC_2025 * (1 - t)
        below = years[traj <= target_inc]
        year_reached[sc][tname] = int(below[0]) if len(below) > 0 else None

# Population weight (India ~1.4B; scale cases to per-100K incidence)
POP = 1_400_000_000
def cases_per_year(traj):
    return traj * POP / 100_000

baseline_traj = trajectories["Baseline"]
cum_averted = {}
for sc in scen_order:
    diff = cases_per_year(baseline_traj) - cases_per_year(trajectories[sc])
    cum_averted[sc] = max(0, np.trapz(diff, years))

# --- Plot ---
fig, axes = plt.subplots(2, 1, figsize=(11, 9),
                         gridspec_kw={"height_ratios": [1.4, 1]})

# Top: trajectories
ax = axes[0]
colors = {"Baseline": "#2c3e50", "ACF": "#2ca02c", "Dx": "#1f77b4",
          "Tx": "#d62728", "ACF+Dx": "#17becf", "ACF+Tx": "#bcbd22",
          "Dx+Tx": "#9467bd", "ACF+Dx+Tx": "#e377c2"}
for sc in scen_order:
    lw = 3.2 if sc == "ACF+Dx+Tx" else 2.0
    ls = "-"
    ax.plot(years, trajectories[sc], color=colors[sc], lw=lw, ls=ls,
            label=sc, zorder=5 if sc == "ACF+Dx+Tx" else 3)


# Deferred label placement: stack labels with sufficient vertical separation
# Label format: "ACF: -20%" combines intervention name with reduction
endpoints = [(sc, float(trajectories[sc][-1])) for sc in scen_order]
endpoints.sort(key=lambda t: -t[1])
MIN_SEP = 11.0  # minimum vertical separation in y-units (per 100k)
prev_y = None
adjusted = []
for sc, y in endpoints:
    if prev_y is not None and prev_y - y < MIN_SEP:
        y_adj = prev_y - MIN_SEP
    else:
        y_adj = y
    adjusted.append((sc, y, y_adj))
    prev_y = y_adj
for sc, y_data, y_label in adjusted:
    pct_reduced = (1 - y_data / INC_2025) * 100
    annot = "  " + sc + ": -" + str(round(pct_reduced)) + "%"
    # Light connector line if label is offset from data point
    if abs(y_label - y_data) > 1.0:
        ax.plot([2035, 2035.15], [y_data, y_label],
                color=colors[sc], lw=0.6, alpha=0.5, zorder=2)
    ax.annotate(annot, xy=(2035, y_data),
                xytext=(2035.2, y_label),
                fontsize=8.5, color=colors[sc], va="center", fontweight="bold")

ax.axhline(INC_2025 * 0.50, color="orange", lw=1.2, ls="--", alpha=0.7,
           label="Interim 50% reduction target")
ax.axhline(INC_2025 * 0.10, color="red", lw=1.5, ls="--", alpha=0.7,
           label="End-TB 2035 target (90%)")

ax.set_xlim(2025, 2037.5)
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("TB incidence (per 100,000)", fontsize=11)
ax.set_title("Path to End-TB 2035: Projected Incidence Trajectories",
             fontsize=12, pad=8)
ax.legend(loc="lower left", fontsize=8.5, ncol=2, framealpha=0.92)
ax.grid(True, alpha=0.3)

# Bottom: cumulative cases averted
ax = axes[1]
sc_for_bar = [s for s in scen_order if s != "Baseline"]
vals = [cum_averted[s] / 1e6 for s in sc_for_bar]
bars = ax.barh(sc_for_bar, vals,
               color=[colors[s] for s in sc_for_bar], alpha=0.85)
for bar, v in zip(bars, vals):
    ax.text(v + max(vals) * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{v:.1f}M", va="center", fontsize=9, fontweight="bold")
ax.invert_yaxis()
ax.set_xlabel("Cumulative cases averted 2025-2035 (millions, vs Baseline)",
              fontsize=11)
ax.set_title("Cumulative Impact: Total Cases Averted Over 10 Years",
             fontsize=12, pad=8)
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
out_path = os.path.join("figures", "FigP1_path_to_endtb.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)
print()
print("Year reaching End-TB 90% target:")
for sc in scen_order:
    print(f"  {sc:12s}: {year_reached[sc]['90% (End-TB)'] or 'not by 2035'}")
print()
print("Cumulative cases averted 2025-2035 (millions):")
for sc in scen_order:
    print(f"  {sc:12s}: {cum_averted[sc] / 1e6:>6.2f}M")


### Policy Figure P2 -- Strategic Priority Matrix

Tabular heatmap summarizing decision-relevant outcomes for all 8 scenarios.
Each cell color-coded green (better) to red (worse). Format favored by WHO and
Global Fund for program evaluation.


In [ ]:
# ======================================================================
# POLICY FIGURE P2: Strategic Priority Matrix (table-as-figure)
# ======================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

if "scenario_results" not in dir():
    raise RuntimeError("Need scenario_results from Figure 5 cell.")

scen_df = (pd.DataFrame(scenario_results)
           if isinstance(scenario_results, list) else scenario_results.copy())
scen_order = ["Baseline", "ACF", "Dx", "Tx",
              "ACF+Dx", "ACF+Tx", "Dx+Tx", "ACF+Dx+Tx"]
scen_df = scen_df.set_index("scenario").reindex(scen_order)

# Build metrics
metrics = pd.DataFrame(index=scen_order)
metrics["Incidence reduction by 2035 (%)"] = scen_df.get(
    "inc_reduction", pd.Series(0, index=scen_order))
# Mortality reduction proxied as 0.85x incidence reduction (typical TB ratio)
metrics["Mortality reduction by 2035 (%)"] = metrics["Incidence reduction by 2035 (%)"] * 0.85

# Year reaching milestones (using same model as P1)
INC_2025 = 210.0
years = np.arange(2025, 2051)
k = 0.30
def year_for_target(inc_red_pct, target_red):
    inc_endpoint = INC_2025 * (1 - inc_red_pct / 100.0)
    if inc_endpoint > INC_2025 * (1 - target_red):
        return None  # never reaches
    traj = inc_endpoint + (INC_2025 - inc_endpoint) * np.exp(-k * (years - 2025))
    target_inc = INC_2025 * (1 - target_red)
    below = years[traj <= target_inc]
    return int(below[0]) if len(below) > 0 else None

metrics["Year reaching 50% reduction"] = [
    year_for_target(scen_df.loc[s].get("inc_reduction", 0), 0.50)
    for s in scen_order]
metrics["Year reaching 90% (End-TB)"] = [
    year_for_target(scen_df.loc[s].get("inc_reduction", 0), 0.90)
    for s in scen_order]

# Loop burden remaining (% of cases in any loop group)
loop_pct = []
for s in scen_order:
    r = scen_df.loc[s]
    pct = float(r.get("TestLooponly", 0)) + float(r.get("TestTxfail", 0))
    loop_pct.append(pct)
metrics["Cases still in loops (%)"] = loop_pct

# Treatment success rate proxy
tx_success = []
for s in scen_order:
    r = scen_df.loc[s]
    txfail_share = float(r.get("TestTxfail", 0))
    tx_success.append(100 - txfail_share * 0.6)  # rough proxy
metrics["Treatment success rate (%)"] = tx_success

# --- Plot as colored table ---
fig, ax = plt.subplots(figsize=(13, 5.5))
cell_text = []
norm_data = np.zeros((len(scen_order), len(metrics.columns)))

for i, sc in enumerate(scen_order):
    row = []
    for j, col in enumerate(metrics.columns):
        v = metrics.loc[sc, col]
        if pd.isna(v) or v is None:
            row.append("not by 2035")
            norm_data[i, j] = -1
        elif "Year reaching" in col:
            row.append(str(int(v)))
            # Earlier is better; normalize: 2025 -> 1.0, 2050 -> 0.0
            norm_data[i, j] = max(0, (2050 - v) / 25.0)
        elif "Cases still in loops" in col:
            row.append(f"{v:.0f}%")
            norm_data[i, j] = 1 - v / max(metrics["Cases still in loops (%)"].max(), 1)
        else:
            row.append(f"{v:.0f}%")
            norm_data[i, j] = v / max(metrics[col].max(), 1)
    cell_text.append(row)

# Color cells
ax.set_xticks(range(len(metrics.columns)))
ax.set_yticks(range(len(scen_order)))
ax.set_xticklabels(metrics.columns, rotation=22, ha="right", fontsize=9)
ax.set_yticklabels(scen_order, fontsize=10)

cmap = LinearSegmentedColormap.from_list(
    "redgreen", ["#fde0e0", "#fff5b3", "#c6e8c6"], N=256)

for i in range(len(scen_order)):
    for j in range(len(metrics.columns)):
        v = norm_data[i, j]
        if v < 0:
            color = "#e0e0e0"
        else:
            color = cmap(v)
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                   facecolor=color, edgecolor="white", lw=2))
        ax.text(j, i, cell_text[i][j], ha="center", va="center",
                fontsize=10, fontweight="bold", color="#222")

ax.set_xlim(-0.5, len(metrics.columns) - 0.5)
ax.set_ylim(-0.5, len(scen_order) - 0.5)
ax.invert_yaxis()
ax.set_title("Figure P2: Strategic Priority Matrix\n"
             "(green = more favorable; red = less favorable)",
             fontsize=12, pad=10)
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
out_path = os.path.join("figures", "FigP2_strategic_matrix.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_path)
print()
print(metrics.to_string())


### Policy Figure P3 -- Pareto Efficient Frontier

Plots each scenario by implementation effort vs. cumulative cases averted.
Pareto-optimal scenarios are connected by a frontier line; dominated scenarios
are shown with reduced opacity. Reveals which bundles are worth the cost.


In [ ]:
# ======================================================================
# POLICY FIGURE P3: Pareto Efficient Frontier
# ======================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "scenario_results" not in dir():
    raise RuntimeError("Need scenario_results.")

scen_df = (pd.DataFrame(scenario_results)
           if isinstance(scenario_results, list) else scenario_results.copy())
scen_order = ["Baseline", "ACF", "Dx", "Tx",
              "ACF+Dx", "ACF+Tx", "Dx+Tx", "ACF+Dx+Tx"]
scen_df = scen_df.set_index("scenario").reindex(scen_order)

# Implementation effort: weighted sum of intervention components
# ACF = 1.5 (high logistical cost), Dx = 1.0 (one-time tech upgrade), Tx = 1.2 (ongoing)
EFFORT_WEIGHTS = {"ACF": 1.5, "Dx": 1.0, "Tx": 1.2}
def effort(name):
    score = 0
    for k, w in EFFORT_WEIGHTS.items():
        if k in name and name != "Baseline":
            score += w
    return score

efforts = np.array([effort(s) for s in scen_order])

# Cumulative cases averted (same model as P1)
INC_2025 = 210.0
POP = 1_400_000_000
years = np.arange(2025, 2036)
k = 0.30

def cum_avert(inc_red_pct):
    inc_endpoint = INC_2025 * (1 - inc_red_pct / 100.0)
    traj = inc_endpoint + (INC_2025 - inc_endpoint) * np.exp(-k * (years - 2025))
    base_traj = np.full_like(years, INC_2025, dtype=float)
    diff = (base_traj - traj) * POP / 100_000
    return max(0, np.trapz(diff, years))

inc_red_col = "inc_reduction" if "inc_reduction" in scen_df.columns else None
cum_averted = np.array([cum_avert(scen_df.loc[s, inc_red_col] if inc_red_col else 0)
                        for s in scen_order]) / 1e6  # millions

# Identify Pareto frontier: increasing effort -> increasing benefit, no point dominated
order_idx = np.argsort(efforts)
pareto_idx = []
best_y = -1
for i in order_idx:
    if cum_averted[i] > best_y:
        pareto_idx.append(i)
        best_y = cum_averted[i]
pareto_set = set(pareto_idx)

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 7))
colors = {"Baseline": "#2c3e50", "ACF": "#2ca02c", "Dx": "#1f77b4",
          "Tx": "#d62728", "ACF+Dx": "#17becf", "ACF+Tx": "#bcbd22",
          "Dx+Tx": "#9467bd", "ACF+Dx+Tx": "#e377c2"}

# Frontier line
front_e = efforts[pareto_idx]
front_b = cum_averted[pareto_idx]
sort = np.argsort(front_e)
ax.plot(front_e[sort], front_b[sort], "-", color="#666", lw=2, alpha=0.6,
        zorder=1, label="Pareto frontier")

# Scenario dots
for i, sc in enumerate(scen_order):
    is_pareto = i in pareto_set
    ax.scatter(efforts[i], cum_averted[i], s=260,
               color=colors[sc], edgecolor="black", lw=1.5,
               alpha=1.0 if is_pareto else 0.45,
               zorder=5 if is_pareto else 3)
    # Annotation
    dx, dy = (0.08, 0.5) if sc != "Baseline" else (0.08, -1.2)
    weight = "bold" if is_pareto else "normal"
    label = sc + (" *" if is_pareto else "")
    ax.annotate(label, (efforts[i], cum_averted[i]),
                xytext=(dx, dy), textcoords="offset fontsize",
                fontsize=10, fontweight=weight, color=colors[sc])

ax.set_xlabel("Implementation effort\n(weighted: ACF=1.5, Dx=1.0, Tx=1.2)",
              fontsize=11)
ax.set_ylabel("Cumulative cases averted 2025-2035 (millions)", fontsize=11)
ax.set_title("Figure P3: Pareto Efficient Frontier of Intervention Bundles\n"
             "(* = Pareto-optimal; faded = dominated)",
             fontsize=12, pad=10)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right", fontsize=10)

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
out_path = os.path.join("figures", "FigP3_pareto_frontier.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)
print()
print("Pareto-optimal scenarios:")
for i in pareto_idx:
    print(f"  {scen_order[i]:12s}: effort={efforts[i]:.1f}, cases_averted={cum_averted[i]:.2f}M")
print("Dominated scenarios:")
for i, sc in enumerate(scen_order):
    if i not in pareto_set:
        print(f"  {sc:12s}: effort={efforts[i]:.1f}, cases_averted={cum_averted[i]:.2f}M (dominated)")


### Policy Figure P7 -- Loop Sensitivity Analysis

If we could **additionally reduce loop burden** (beyond what the intervention itself
removes), how would each of the 8 scenarios scale toward End-TB targets? This
sensitivity sweeps a hypothetical "loop reduction multiplier" from 0% (current
high-loop baseline) to 80% (near-ideal cascade).

Reveals which scenarios are **loop-bottlenecked** (steep slope: gain a lot from
loop repair) vs **loop-saturated** (flat: already near ceiling). Also shows the
loop-reduction threshold required for each scenario to cross the End-TB 2035 target.


In [ ]:
# ======================================================================
# POLICY FIGURE P7: Loop Sensitivity Analysis
# How each scenario's incidence reduction scales with additional loop reduction
# ======================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "scenario_results" not in dir() or "episode_df" not in dir():
    raise RuntimeError("Need scenario_results and episode_df.")

scen_df = (pd.DataFrame(scenario_results)
           if isinstance(scenario_results, list) else scenario_results.copy())
scen_order = ["Baseline", "ACF", "Dx", "Tx",
              "ACF+Dx", "ACF+Tx", "Dx+Tx", "ACF+Dx+Tx"]
scen_df = scen_df.set_index("scenario").reindex(scen_order)

# Baseline loop share of total infectious time (from episode_df)
loop_cols = ["screening_loop_days", "diagnostic_loop_days", "treatment_loop_days"]
total_loop = float(episode_df[loop_cols].sum().sum())
total_all  = float(episode_df.sum().sum())
baseline_loop_share = total_loop / total_all
print("Baseline loop share of all infectious time: %.1f%%" % (baseline_loop_share * 100))

inc_red_col = "inc_reduction" if "inc_reduction" in scen_df.columns else "inc_reduction_pct"
R_AMP = 1.5  # endemic feedback multiplier (conservative for TB cascade dynamics) (consistent with Fig 5 scenarios cell)

# Sweep: additional loop reduction beyond what the intervention itself does
loop_reduction_grid = np.linspace(0.0, 0.80, 50)

# For each scenario, recompute incidence reduction at each loop_reduction level.
# Model: total direct reduction = scenario_direct + L * baseline_loop_share
# Then amplified: inc_red = 1 - (1 - clamp(total))**R_AMP
def scenario_curve(sc):
    inc_red_baseline = scen_df.loc[sc, inc_red_col] / 100.0
    # Recover the "direct reduction" implied by the amplified value:
    # inc_red = 1 - (1 - direct)^R_AMP -> direct = 1 - (1 - inc_red)^(1/R_AMP)
    direct_baseline = 1 - max(1e-6, (1 - inc_red_baseline)) ** (1.0 / R_AMP)
    curve = []
    for L in loop_reduction_grid:
        # Each unit of L removes L * baseline_loop_share of total infectious time
        # (additive on top of intervention's direct effect)
        d = min(direct_baseline + L * baseline_loop_share, 0.95)
        ir = 1 - (1 - d) ** R_AMP
        curve.append(ir * 100)
    return np.array(curve)

curves = {sc: scenario_curve(sc) for sc in scen_order}

# Find the L threshold at which each scenario crosses 90% incidence reduction
def threshold(sc, target_pct=90.0):
    c = curves[sc]
    above = loop_reduction_grid[c >= target_pct]
    return float(above[0]) if len(above) > 0 else None

thresholds_50 = {sc: threshold(sc, 50.0) for sc in scen_order}
thresholds_90 = {sc: threshold(sc, 90.0) for sc in scen_order}

# --- Plot ---
fig, ax = plt.subplots(figsize=(11.5, 7.0))
colors = {"Baseline": "#2c3e50", "ACF": "#2ca02c", "Dx": "#1f77b4",
          "Tx": "#d62728", "ACF+Dx": "#17becf", "ACF+Tx": "#bcbd22",
          "Dx+Tx": "#9467bd", "ACF+Dx+Tx": "#e377c2"}

for sc in scen_order:
    lw = 3.2 if sc == "ACF+Dx+Tx" else 2.0
    ax.plot(loop_reduction_grid * 100, curves[sc],
            color=colors[sc], lw=lw, label=sc,
            zorder=5 if sc == "ACF+Dx+Tx" else 3)

# Target lines
ax.axhline(50, color="orange", lw=1.2, ls="--", alpha=0.7,
           label="Interim target (-50%)")
ax.axhline(90, color="red", lw=1.5, ls="--", alpha=0.7,
           label="End-TB 2035 target (-90%)")

# Threshold markers (where each line crosses 90%)
for sc in scen_order:
    t = thresholds_90[sc]
    if t is not None:
        ax.axvline(t * 100, color=colors[sc], lw=0.8, ls=":", alpha=0.5)
        ax.scatter([t * 100], [90], s=80, color=colors[sc], zorder=10,
                   edgecolor="white", lw=1.2)

ax.set_xlabel("Additional loop reduction (% of baseline loop burden removed)",
              fontsize=11)
ax.set_ylabel("Incidence reduction by 2035 (%)", fontsize=11)
ax.set_title("Figure P7: Loop Sensitivity Analysis\n"
             "How each scenario scales when loop burden is reduced further",
             fontsize=12, pad=10)
ax.legend(loc="lower right", fontsize=8.5, ncol=2, framealpha=0.95)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 80)
ax.set_ylim(0, 100)

# Annotation for current state
ax.text(2, 5, "Current high-loop\nbaseline", fontsize=9, color="grey",
        style="italic")

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
out_path = os.path.join("figures", "FigP7_loop_sensitivity.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)
print()
print("Loop reduction needed to cross 50%% / 90%% incidence reduction targets:")
for sc in scen_order:
    t50 = thresholds_50[sc]
    t90 = thresholds_90[sc]
    s50 = ("%.0f%%" % (t50 * 100)) if t50 is not None else "not within 80%"
    s90 = ("%.0f%%" % (t90 * 100)) if t90 is not None else "not within 80%"
    print("  %-12s 50%%: %-15s 90%%: %s" % (sc, s50, s90))


#### Figure 6 -- Impact Curves: Intervention Effectiveness

How incidence reduction depends on loop reduction intensity.
Combined interventions show super-additive effects because they
address different loop types simultaneously.
